In [ ]:
print("ram ram")

# Initialization

In [ ]:
import os
import sys
from dotenv import load_dotenv, find_dotenv
import logging
from typing import Annotated, List, Dict, Any, Literal
from typing_extensions import TypedDict
import litellm
from langchain_litellm import ChatLiteLLM  
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
import re
from langchain_core.tools import tool
from tavily import TavilyClient
from langgraph.prebuilt import ToolNode
from sqlalchemy import create_engine, text
import requests
from bs4 import BeautifulSoup
import uuid
from pydantic import BaseModel, Field
from pinecone import Pinecone
import json
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import time
load_dotenv(find_dotenv(), override=True) # Forces Jupyter to re-read your system keys fresh


In [ ]:
from typing import Annotated, Any, Dict, List, Optional
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class TeamState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next_agent: str
    source_documents: List[Dict[str, Any]]
    generated_outputs: List[Dict[str, Any]]
    user_id: str
    current_investigated_date: Optional[str]
    date_summary_cache: Optional[str]
    current_video_camera: Optional[str]
    video_summary_cache: Optional[str]


In [ ]:
from typing import Annotated, Any, Dict, List, Optional
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class TeamState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next_agent: str
    source_documents: List[Dict[str, Any]]
    generated_outputs: List[Dict[str, Any]]
    user_id: str
    current_investigated_date: Optional[str]
    date_summary_cache: Optional[str]


In [ ]:
import json
import time
import os
import re
import litellm
from langchain_litellm import ChatLiteLLM
from langchain_core.messages import AIMessage

# ====================================================================
# 🤫 SILENCE LITELLM LOGS COMPLETELY
# ====================================================================
litellm.set_verbose = False
litellm.suppress_debug_info = True
litellm.turn_off_message_logging = True

try:
    litellm._logging._disable_debugging()
except AttributeError:
    pass

# ====================================================================
# INITIALIZE LITELLM PROVIDERS
# ====================================================================
groq_llm = ChatLiteLLM(
    model="groq/llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1,
    max_tokens=1500,
)

gemini_llm = ChatLiteLLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.1,
    max_tokens=1500,
)

# ====================================================================
# PRODUCTION LLM GATEWAY & FALLBACK ORCHESTRATION LAYER 🚀
# ====================================================================
class FallbackLLM:

    def __init__(self, models):
        self.models = models
        # Global Registry for tracking gateway performance over time
        self.gateway_metrics = {
            "total_calls": 0,
            "failed_calls": 0,
            "input_tokens_served": 0,
            "output_tokens_served": 0,
            "accumulated_cost_usd": 0.0
        }
        # Industry price points per 1M tokens (Standard 2026 Registry limits)
        self._cost_table = {
            "groq/llama-3.1-8b-instant": {"input": 0.59, "output": 0.79},
            "gemini/gemini-3.1-flash-lite": {"input": 0.075, "output": 0.30},
        }

    def _input_guardrail(self, messages) -> bool:
        """Level 1 Guardrail: Scans the incoming conversation stream."""
        if not messages:
            return True
            
        last_msg = messages[-1]
        content = getattr(last_msg, "content", "")
        if not isinstance(content, str):
            return True
            
        content_lower = content.lower()
        
        blocked_patterns = [
            "drop table", "delete from", "truncate table", "insert into",
            "ignore previous instructions", "system override", "reveal your system prompt"
        ]
        
        for pattern in blocked_patterns:
            if pattern in content_lower:
                return False
        return True

    def _output_guardrail(self, content: str) -> str:
        """Level 3 Guardrail: Stops raw API keys or passwords from rendering."""
        if not content:
            return content
            
        # 1. Intercept Raw Database Strings
        db_url_pattern = r"(postgresql|postgres|mysql|mongodb):\/\/([^:]+):([^@]+)@([^/]+)\/([^?\s]+)"
        if re.search(db_url_pattern, content):
            return "⚠️ [SECURITY ENFORCEMENT]: Sensitive database credentials were intercepted and hidden."
            
        # 2. Intercept Secret Encryption Column Names
        sensitive_keywords = [
            "password_hash", "encrypted_app_password", "encrypted_password", 
            "app_password", "gemini_key", "groq_key", "openrouter_key", "tavily_key"
        ]
        for keyword in sensitive_keywords:
            if keyword in content.lower():
                return "⚠️ [SECURITY ENFORCEMENT]: Response blocked to prevent raw encryption keys or passwords from showing."
                
        return content

    def _calculate_costs(self, model_name: str, input_tokens: int, output_tokens: int):
        """Calculates exact API usage costs based on provider rates."""
        if model_name in self._cost_table:
            rates = self._cost_table[model_name]
            in_cost = (input_tokens / 1_000_000) * rates["input"]
            out_cost = (output_tokens / 1_000_000) * rates["output"]
            total_call_cost = in_cost + out_cost
            
            # Record state changes inside the gateway instance memory
            self.gateway_metrics["input_tokens_served"] += input_tokens
            self.gateway_metrics["output_tokens_served"] += output_tokens
            self.gateway_metrics["accumulated_cost_usd"] += total_call_cost

    def invoke(self, messages, **kwargs):
        self.gateway_metrics["total_calls"] += 1
        
        # --- Level 1 Input Guardrail Integration ---
        if not self._input_guardrail(messages):
            return AIMessage(content="🚨 [SECURITY VIOLATION]: Request blocked by system gateway.")

        last_error = None

        for provider_name, llm in self.models:
            try:
                start = time.perf_counter()
                response = llm.invoke(messages, **kwargs)
                elapsed = time.perf_counter() - start

                # Safely check if response is a structured output (Pydantic model or dict)
                from pydantic import BaseModel
                is_structured = isinstance(response, BaseModel) or isinstance(response, dict)

                # Extrapolate structural metadata values safely
                input_tokens = 0
                output_tokens = 0
                if not is_structured:
                    if hasattr(response, "usage_metadata") and response.usage_metadata:
                        input_tokens = response.usage_metadata.get("input_tokens", 0)
                        output_tokens = response.usage_metadata.get("output_tokens", 0)
                    elif hasattr(response, "response_metadata") and response.response_metadata:
                        usage = response.response_metadata.get("token_usage") or response.response_metadata.get("usage") or {}
                        input_tokens = usage.get("prompt_tokens", 0)
                        output_tokens = usage.get("completion_tokens", 0)

                    # Execute cost registry modifications
                    model_str = getattr(llm, "model", "")
                    self._calculate_costs(model_str, input_tokens, output_tokens)

                    # --- Level 3 Output Guardrail Filter Engine ---
                    if hasattr(response, "content") and response.content:
                        response.content = self._output_guardrail(response.content)

                    if hasattr(response, "response_metadata") and response.response_metadata is not None:
                        response.response_metadata["selected_provider"] = provider_name
                        response.response_metadata["response_time"] = elapsed
                return response

            except Exception as e:
                self.gateway_metrics["failed_calls"] += 1
                last_error = e
                print(f"⚠️ Gateway Notice: {provider_name} fallback activated. Redirecting network traffic...")
                continue

        raise RuntimeError("All configured LLM providers are unavailable.") from last_error

    def with_structured_output(self, schema, **kwargs):
        structured = []
        for provider_name, llm in self.models:
            structured.append((provider_name, llm.with_structured_output(schema, **kwargs)))
        return FallbackLLM(structured)

    def bind_tools(self, tools):
        bound = []
        for provider_name, llm in self.models:
            bound.append((provider_name, llm.bind_tools(tools)))
        return FallbackLLM(bound)

# ====================================================================
# INSTANTIATE GLOBAL GATEWAY ASSETS
# ====================================================================
# Swap the order of models inside the gateway initialization cell
base_llm = FallbackLLM([
    ("Gemini", gemini_llm),  # Gemini becomes the primary driver
    ("Groq", groq_llm),      # Groq acts as the backup fallback layer
])

search_planner_llm = base_llm
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


# DB Connection

In [ ]:
import os
import time
from sqlalchemy import create_engine, text
from sqlalchemy.pool import QueuePool
from dotenv import load_dotenv

load_dotenv()

DATABASE_URL = os.getenv("CONSTRUCTION_DB_URL")

if not DATABASE_URL:
    raise ValueError("CONSTRUCTION_DB_URL is not set in the environment.")

# ====================================================================
# 🗄️ PRODUCTION-GRADE DATABASE ENGINE (Pooled + Self-Healing)
# ====================================================================
engine = None
_DB_MAX_RETRIES = 3
_DB_RETRY_DELAY_SECONDS = 2

for attempt in range(1, _DB_MAX_RETRIES + 1):
    try:
        engine = create_engine(
            DATABASE_URL,
            poolclass=QueuePool,
            pool_size=10,            # persistent connections kept warm for agent tool calls
            max_overflow=20,         # burst capacity under concurrent multi-agent load
            pool_timeout=30,         # seconds to wait for a free connection before erroring
            pool_recycle=1800,       # recycle connections every 30 min (avoids stale TCP drops)
            pool_pre_ping=True,      # validates connection liveness before every checkout
            connect_args={"connect_timeout": 10},
        )

        with engine.connect() as conn:
            print(f"✅ Database connected successfully! (attempt {attempt}/{_DB_MAX_RETRIES})")

            table_exists = conn.execute(text("""
                SELECT EXISTS (
                    SELECT 1
                    FROM information_schema.tables
                    WHERE table_schema = 'public'
                      AND table_name = 'users'
                );
            """)).scalar()

            if table_exists:
                user_count = conn.execute(text("SELECT COUNT(*) FROM users;")).scalar()
                camera_count = conn.execute(text("SELECT COUNT(*) FROM cameras;")).scalar()
                print("✅ 'users' table found.")
                print(f"👥 Total Users: {user_count}")
                print(f"📷 Total Cameras: {camera_count}")
            else:
                print("❌ 'users' table does not exist.")
        break

    except Exception as e:
        print(f"❌ Database connection attempt {attempt}/{_DB_MAX_RETRIES} failed: {e}")
        engine = None
        if attempt < _DB_MAX_RETRIES:
            time.sleep(_DB_RETRY_DELAY_SECONDS)
        else:
            print("❌ Database connection failed after all retries. Tools will report a clear error instead of crashing.")


def get_db_health() -> dict:
    """Lightweight health probe used by ops/monitoring and the System Agent."""
    if engine is None:
        return {"status": "down", "error": "Engine not initialized."}
    try:
        start = time.perf_counter()
        with engine.connect() as conn:
            conn.execute(text("SELECT 1;"))
        latency_ms = round((time.perf_counter() - start) * 1000, 2)
        pool = engine.pool
        return {
            "status": "healthy",
            "latency_ms": latency_ms,
            "pool_checked_out": pool.checkedout(),
            "pool_size": pool.size(),
        }
    except Exception as e:
        return {"status": "down", "error": str(e)}


# General Agent

In [ ]:
from sqlalchemy import create_engine, text
from langchain_core.tools import tool
from typing import Dict, Any

# ====================================================================
# 🛠️ GENERAL AGENT SANDBOXED USER DETAILS TOOL
# ====================================================================

@tool
def get_current_user_profile(user_email: str) -> Dict[str, Any]:
    """
    USER DETAILS EXPLORER: Looks up complete administrative profile settings, campus department mapping, 
    designation hierarchy description data, and permissions properties for a specific system operator.
    
    Use this when the user says:
      - 'Show my profile details'
      - 'Who am I logged in as?'
      - 'Check my department registration status'
    """
    if engine is None: 
        return {"success": False, "error": "Database connector engine is uninitialized."}
        
    query = text("""
        SELECT 
            u.id AS user_id, 
            u.full_name, 
            u.email, 
            u.username, 
            u.phone, 
            u.employee_id, 
            u.is_active, 
            u.last_login,
            r.name AS platform_role_name,
            d.name AS department_name, 
            d.description AS department_description,
            des.name AS designation_title
        FROM users u
        LEFT JOIN roles r ON r.id = u.role_id
        LEFT JOIN departments d ON d.id = u.department_id
        LEFT JOIN designations des ON des.id = u.designation_id
        WHERE u.email = :email OR u.username = :email;
    """)
    
    try:
        with engine.connect() as conn:
            row = conn.execute(query, {"email": str(user_email)}).fetchone()
            if not row:
                return {"success": False, "error": f"Operator profile linked to parameter '{user_email}' not found."}
                
            data = dict(row._mapping)
            
            # Serialize timestamps cleanly into standard plain-text strings for the LLM context
            if data.get("last_login"):
                data["last_login"] = str(data["last_login"])
                
            return {
                "success": True,
                "profile": data
            }
            
    except Exception as e:
        return {"success": False, "error": f"Failed to retrieve operator identity parameters: {str(e)}"}

# Global assignment array explicitly tracking the single General Agent user tool component
general_agent_tools_registry = [
    get_current_user_profile
]


# System agent

In [ ]:
from sqlalchemy import create_engine, text
from langchain_core.tools import tool
from typing import List, Dict, Any, Optional, Literal

# ====================================================================
# 🛡️ SHARED SECURITY UTILITY ENGINE
# ====================================================================
def _is_query_secure(sql_query: str) -> tuple[bool, str]:
    """Internal validation mechanism to protect core database systems."""
    query_lower = sql_query.lower()
    
    # 1. Block destructive operations (SQL injection protection)
    forbidden_ops = ["insert", "update", "delete", "drop", "truncate", "alter", "grant", "create table"]
    if any(cmd in query_lower for cmd in forbidden_ops):
        return False, "Security Exception: Non-read command strings are explicitly barred inside this node block."
        
    # 2. Block sensitive security rows (Credential protection)
    forbidden_columns = ["password_hash", "smtp_password", "telegram_bot_token", "teams_webhook_url", "whatsapp_auth_token"]
    if any(col in query_lower for col in forbidden_columns):
        return False, "Security Exception: Reading structural security parameters or api tokens via chat is restricted."
        
    return True, ""

def _serialize_row_data(result) -> List[Dict[str, Any]]:
    """Transforms raw engine row mappings into primitive types consumable by LLM frames."""
    forbidden_columns = ["password_hash", "smtp_password", "telegram_bot_token", "teams_webhook_url", "whatsapp_auth_token"]
    serialized_rows = []
    
    for row in result:
        d = dict(row._mapping)
        for k, v in d.items():
            # Double-layer protection: strip sensitive parameters if a wildcard was used
            if k.lower() in forbidden_columns:
                d[k] = "[ENCRYPTED_REDACTED]"
            elif v is not None and not isinstance(v, (int, float, str, bool, list, dict)):
                d[k] = str(v) # Safely cast datetimes, uuids, and decimals to text
        serialized_rows.append(d)
    return serialized_rows


# ====================================================================
# 🛠️ SYSTEM AGENT READ-ONLY TOOLS
# ====================================================================

@tool
def query_system_data(sql_query: str) -> List[Dict[str, Any]]:
    """
    DYNAMIC QUERY EXECUTOR: Use this to read telemetry parameters, user rosters, zones, 
    or activity metrics from the industrial vision database.
    
    Examples:
      - 'SELECT id, name, status FROM cameras;'
      - 'SELECT channel, status, created_at FROM notification_logs ORDER BY created_at DESC LIMIT 5;'
      - 'SELECT * FROM zones WHERE is_active = true;'
    """
    if engine is None: 
        return [{"error": "Database connector engine is uninitialized."}]
        
    is_secure, error_msg = _is_query_secure(sql_query)
    if not is_secure:
        return [{"error": error_msg}]
        
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql_query))
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": f"SQL Execution Failed: {str(e)}"}]


@tool
def check_camera_fleet_health() -> List[Dict[str, Any]]:
    """
    HIGH-SPEED MACRO: Fetches the connection status of all video recording components, 
    including Basler devices, alongside their last recorded network logs.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    
    query = """
        SELECT 
            c.id AS camera_id, c.name, c.ip, c.status AS current_status,
            (SELECT log.status FROM camera_status_logs log WHERE log.camera_id = c.id ORDER BY log.checked_at DESC LIMIT 1) AS last_logged_status,
            (SELECT log.checked_at FROM camera_status_logs log WHERE log.camera_id = c.id ORDER BY log.checked_at DESC LIMIT 1) AS last_check_time
        FROM cameras c
        ORDER BY c.status ASC, c.id DESC;
    """
    try:
        with engine.connect() as conn:
            result = conn.execute(text(query))
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def fetch_active_safety_alerts(limit: int = 10, severity_filter: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    HIGH-SPEED MACRO: Retrieves recent high-priority security, HSE rule violations, or 
    unacknowledged alerts from live video tracking streams.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    
    base_sql = """
        SELECT a.id, a.camera_name, a.class_name, a.confidence, a.is_acknowledged, a.created_at,
               z.name AS zone_name, z.zone_type
        FROM alerts a
        LEFT JOIN zones z ON z.id = a.zone_id
        WHERE a.is_acknowledged = false
    """
    params = {"limit": limit}
    if severity_filter:
        base_sql += " AND z.zone_type = :severity"
        params["severity"] = severity_filter
        
    base_sql += " ORDER BY a.created_at DESC LIMIT :limit;"
    
    try:
        with engine.connect() as conn:
            result = conn.execute(text(base_sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_production_counting_summary(window_hours: int = 24) -> List[Dict[str, Any]]:
    """
    HIGH-SPEED MACRO: Compiles aggregate manufacturing throughput and conveyor volume analysis 
    computed across your active processing stations over a specific hourly time window.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    
    query = """
        SELECT config_name, SUM(count_in) AS total_in, SUM(count_out) AS total_out, SUM(total_count) AS aggregate_throughput,
               COUNT(id) AS batches_processed
        FROM counting_batches
        WHERE start_time >= NOW() - INTERVAL '1 hour' * :hours
        GROUP BY config_name
        ORDER BY aggregate_throughput DESC;
    """
    try:
        with engine.connect() as conn:
            result = conn.execute(text(query), {"hours": window_hours})
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_cameras(camera_id_or_name: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Retrieves details of cameras, including online/offline status, IP addresses, ports, and camera numbers.
    Can filter by a specific camera using its ID, name, or camera number.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = "SELECT id, name, ip, port, camera_number, status FROM cameras"
    params = {}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " WHERE id = :cid OR camera_number = :cid_str OR name ILIKE :name_like"
            params = {"cid": cid, "cid_str": str(cid), "name_like": f"%{camera_id_or_name}%"}
        except ValueError:
            sql += " WHERE camera_number = :val OR name ILIKE :name_like"
            params = {"val": camera_id_or_name, "name_like": f"%{camera_id_or_name}%"}
    sql += " ORDER BY id;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_zones(camera_id_or_name: Optional[str] = None, zone_name_or_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Retrieves zones, including Red Zones, restricted zones, and other ROIs.
    Can filter by camera name/ID, and/or zone name/type.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT z.id, z.camera_id, c.name as camera_name, z.name, z.zone_type, z.is_active, z.description
        FROM zones z
        LEFT JOIN cameras c ON c.id = z.camera_id
        WHERE 1=1
    """
    params = {}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " AND (z.camera_id = :cid OR c.camera_number = :cid_str OR c.name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cid_str"] = str(cid)
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            sql += " AND (c.camera_number = :cam_val OR c.name ILIKE :cam_name_like)"
            params["cam_val"] = camera_id_or_name
            params["cam_name_like"] = f"%{camera_id_or_name}%"
    if zone_name_or_type is not None:
        sql += " AND (z.name ILIKE :zone_like OR z.zone_type ILIKE :zone_like)"
        params["zone_like"] = f"%{zone_name_or_type}%"
    sql += " ORDER BY z.id;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_incidents(
    camera_id_or_name: Optional[str] = None,
    date_filter: Optional[Literal["today", "yesterday", "this_week"]] = None,
    status: Optional[Literal["active", "resolved", "recurring"]] = None,
    class_name: Optional[str] = None,
    limit: int = 100
) -> List[Dict[str, Any]]:
    """
    Retrieves safety/production incidents. Can filter by camera, date (today, yesterday, this_week),
    status (active, resolved, recurring), and class name (e.g., fire, smoke, helmet, ppe, person, vehicle, forklift).
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT i.id, i.camera_id, i.camera_name, i.zone_id, z.name as zone_name, i.class_name, i.confidence,
               i.started_at, i.resolved_at, i.is_active, i.is_acknowledged, i.is_recurring, i.classification,
               i.escalation_status, i.root_cause
        FROM incidents i
        LEFT JOIN zones z ON z.id = i.zone_id
        WHERE 1=1
    """
    params = {"limit": limit}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " AND (i.camera_id = :cid OR i.camera_name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            sql += " AND i.camera_name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
            
    if date_filter == "today":
        sql += " AND i.started_at >= CURRENT_DATE"
    elif date_filter == "yesterday":
        sql += " AND i.started_at >= CURRENT_DATE - INTERVAL '1 day' AND i.started_at < CURRENT_DATE"
    elif date_filter == "this_week":
        sql += " AND i.started_at >= DATE_TRUNC('week', CURRENT_DATE)"
        
    if status == "active":
        sql += " AND i.is_active = true"
    elif status == "resolved":
        sql += " AND i.resolved_at IS NOT NULL"
    elif status == "recurring":
        sql += " AND i.is_recurring = true"
        
    if class_name is not None:
        sql += " AND i.class_name ILIKE :class_name"
        params["class_name"] = f"%{class_name}%"
        
    sql += " ORDER BY i.started_at DESC LIMIT :limit;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_alerts(
    camera_id_or_name: Optional[str] = None,
    date_filter: Optional[Literal["today", "yesterday"]] = None,
    status: Optional[Literal["active", "all"]] = None,
    priority: Optional[Literal["critical", "high", "all"]] = None,
    class_name: Optional[str] = None,
    limit: int = 100
) -> List[Dict[str, Any]]:
    """
    Retrieves system alerts. Supports filtering by camera name/ID, date range (today, yesterday),
    status (active/unacknowledged), priority (critical/high, matching zones type), and class name.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT a.id, a.camera_id, a.camera_name, a.zone_id, z.name as zone_name, z.zone_type, a.class_name,
               a.confidence, a.is_acknowledged, a.created_at, a.snapshot_path
        FROM alerts a
        LEFT JOIN zones z ON z.id = a.zone_id
        WHERE 1=1
    """
    params = {"limit": limit}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " AND (a.camera_id = :cid OR a.camera_name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            sql += " AND a.camera_name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
            
    if date_filter == "today":
        sql += " AND a.created_at >= CURRENT_DATE"
    elif date_filter == "yesterday":
        sql += " AND a.created_at >= CURRENT_DATE - INTERVAL '1 day' AND a.created_at < CURRENT_DATE"
        
    if status == "active":
        sql += " AND a.is_acknowledged = false"
        
    if priority == "critical":
        sql += " AND z.zone_type ILIKE '%critical%'"
    elif priority == "high":
        sql += " AND (z.zone_type ILIKE '%high%' OR z.zone_type ILIKE '%red%' OR z.zone_type ILIKE '%restricted%')"
        
    if class_name is not None:
        sql += " AND a.class_name ILIKE :class_name"
        params["class_name"] = f"%{class_name}%"
        
    sql += " ORDER BY a.created_at DESC LIMIT :limit;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_anomalies(
    camera_id_or_name: Optional[str] = None,
    date_filter: Optional[Literal["today"]] = None,
    severity: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Retrieves recorded camera or zone anomalies. Supports filtering by camera, severity, and date.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT af.id, af.anomaly_type, af.severity, af.description, af.zone_id, z.name as zone_name,
               af.camera_id, c.name as camera_name, af.class_name, af.baseline_value, af.observed_value,
               af.deviation_factor, af.event_count, af.is_acknowledged, af.created_at
        FROM anomaly_flags af
        LEFT JOIN zones z ON z.id = af.zone_id
        LEFT JOIN cameras c ON c.id = af.camera_id
        WHERE 1=1
    """
    params = {}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " AND (af.camera_id = :cid OR c.name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            sql += " AND c.name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
    if date_filter == "today":
        sql += " AND af.created_at >= CURRENT_DATE"
    if severity is not None:
        sql += " AND af.severity ILIKE :severity"
        params["severity"] = severity
    sql += " ORDER BY af.created_at DESC;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_zone_risk_scores(camera_id_or_name: Optional[str] = None, zone_name: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Retrieves risk score details for zones and cameras.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT zrs.id, zrs.zone_id, z.name as zone_name, zrs.camera_id, c.name as camera_name,
               zrs.risk_score, zrs.risk_level, zrs.event_count, zrs.high_severity_count,
               zrs.recurrence_count, zrs.top_class, zrs.factors, zrs.computed_at
        FROM zone_risk_scores zrs
        LEFT JOIN zones z ON z.id = zrs.zone_id
        LEFT JOIN cameras c ON c.id = zrs.camera_id
        WHERE 1=1
    """
    params = {}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " AND (zrs.camera_id = :cid OR c.name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            sql += " AND c.name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
    if zone_name is not None:
        sql += " AND z.name ILIKE :zone_name"
        params["zone_name"] = f"%{zone_name}%"
    sql += " ORDER BY zrs.risk_score DESC;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_hse_rule_violations(
    camera_id_or_name: Optional[str] = None,
    date_filter: Optional[Literal["today"]] = None,
    rule_name: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Retrieves HSE rule violation events. Supports camera, date, and rule name filtering.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT hre.id, hre.camera_id, c.name as camera_name, hre.rule_id, hrd.name as rule_name,
               hrd.description as rule_description, hre.triggered_at, hre.severity, hre.snapshot_path, hre.detail
        FROM hse_rule_events hre
        LEFT JOIN cameras c ON c.id = hre.camera_id
        LEFT JOIN hse_rule_definitions hrd ON hrd.id = hre.rule_id
        WHERE 1=1
    """
    params = {}
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            sql += " AND (hre.camera_id = :cid OR c.name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            sql += " AND c.name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
    if date_filter == "today":
        sql += " AND hre.triggered_at >= CURRENT_DATE"
    if rule_name is not None:
        sql += " AND hrd.name ILIKE :rule_name"
        params["rule_name"] = f"%{rule_name}%"
    sql += " ORDER BY hre.triggered_at DESC;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_defect_detections(
    date_filter: Optional[Literal["today"]] = None,
    model_name: Optional[str] = None,
    class_name: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Retrieves production defect detections (such as missing screws or screw body defects).
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT dd.id, dd.basler_camera_id, bd.name as basler_camera_name, dd.model_id, dd.model_name,
               dd.class_name, dd.confidence, dd.bbox, dd.image_path, dd.created_at
        FROM defect_detections dd
        LEFT JOIN basler_devices bd ON bd.id = dd.basler_camera_id
        WHERE 1=1
    """
    params = {}
    if date_filter == "today":
        sql += " AND dd.created_at >= CURRENT_DATE"
    if model_name is not None:
        sql += " AND dd.model_name ILIKE :model_name"
        params["model_name"] = f"%{model_name}%"
    if class_name is not None:
        sql += " AND dd.class_name ILIKE :class_name"
        params["class_name"] = f"%{class_name}%"
    sql += " ORDER BY dd.created_at DESC LIMIT 100;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_counting_statistics(date_filter: Optional[Literal["today"]] = None) -> List[Dict[str, Any]]:
    """
    Retrieves production counting throughput and conveyor statistics.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT cs.id, cs.config_id, cs.config_name, cs.snapshot_date, cs.total_count, cs.count_in, cs.count_out
        FROM counting_snapshots cs
        WHERE 1=1
    """
    params = {}
    if date_filter == "today":
        sql += " AND cs.snapshot_date >= CURRENT_DATE"
    sql += " ORDER BY cs.snapshot_date DESC, cs.id DESC LIMIT 100;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_recording_history(date_filter: Optional[Literal["today"]] = None, status: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Retrieves video recording history for counting conveyors.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT cr.id, cr.config_id, cr.config_name, cr.batch_id, cr.recording_type,
               cr.folder_date, cr.file_path, cr.start_time, cr.end_time, cr.duration_seconds, cr.status, cr.created_at
        FROM counting_recordings cr
        WHERE 1=1
    """
    params = {}
    if date_filter == "today":
        sql += " AND cr.created_at >= CURRENT_DATE"
    if status is not None:
        sql += " AND cr.status ILIKE :status"
        params["status"] = status
    sql += " ORDER BY cr.created_at DESC LIMIT 100;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_notification_history(
    channel: Optional[str] = None,
    status: Optional[str] = None,
    date_filter: Optional[Literal["today"]] = None
) -> List[Dict[str, Any]]:
    """
    Retrieves log files and delivery history of notifications (e.g. Email, Telegram).
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT nl.id, nl.rule_id, nr.name as rule_name, nl.channel, nl.recipient, nl.status, nl.response, nl.created_at
        FROM notification_logs nl
        LEFT JOIN notification_rules nr ON nr.id = nl.rule_id
        WHERE 1=1
    """
    params = {}
    if channel is not None:
        sql += " AND nl.channel ILIKE :channel"
        params["channel"] = channel
    if status is not None:
        sql += " AND nl.status ILIKE :status"
        params["status"] = status
    if date_filter == "today":
        sql += " AND nl.created_at >= CURRENT_DATE"
    sql += " ORDER BY nl.created_at DESC LIMIT 100;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_scheduled_reports() -> List[Dict[str, Any]]:
    """
    Retrieves all currently active or inactive scheduled reports configurations.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT sr.id, sr.channels, sr.email_recipients, sr.telegram_chat_ids, sr.teams_webhook_url,
               sr.whatsapp_numbers, sr.frequency, sr.day_of_week, sr.day_of_month, sr.send_time,
               sr.format, sr.date_range, sr.class_name, sr.camera_id, c.name as camera_name,
               sr.zone_id, z.name as zone_name, sr.is_active, sr.created_at, sr.last_sent_at
        FROM scheduled_reports sr
        LEFT JOIN cameras c ON c.id = sr.camera_id
        LEFT JOIN zones z ON z.id = sr.zone_id
        ORDER BY sr.id;
    """
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_system_settings() -> List[Dict[str, Any]]:
    """
    Retrieves the overall configuration and security settings of the system.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT id, archive_days, auto_delete_low_severity, storage_location,
               session_timeout, enforce_2fa, api_token_expiry, ip_allowlist, created_at, updated_at
        FROM system_settings;
    """
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_users_and_activity(username: Optional[str] = None, date_filter: Optional[Literal["today"]] = None) -> List[Dict[str, Any]]:
    """
    Retrieves registered users and their platform activity history logs (e.g. login events).
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT ual.id, ual.user_id, u.username, u.full_name, u.email, ual.action, ual.detail, ual.ip_address, ual.created_at
        FROM user_activity_logs ual
        LEFT JOIN users u ON u.id = ual.user_id
        WHERE 1=1
    """
    params = {}
    if username is not None:
        sql += " AND (u.username ILIKE :username OR u.full_name ILIKE :username)"
        params["username"] = f"%{username}%"
    if date_filter == "today":
        sql += " AND ual.created_at >= CURRENT_DATE"
    sql += " ORDER BY ual.created_at DESC LIMIT 100;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def get_ai_recommendations(date_filter: Optional[Literal["today"]] = None) -> List[Dict[str, Any]]:
    """
    Retrieves actionable recommendations generated by AI agents regarding security or safety.
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    sql = """
        SELECT ar.id, ar.agent_id, ar.category, ar.priority, ar.title, ar.description,
               ar.zone_id, z.name as zone_name, ar.camera_id, c.name as camera_name,
               ar.is_acknowledged, ar.is_dismissed, ar.created_at
        FROM agent_recommendations ar
        LEFT JOIN zones z ON z.id = ar.zone_id
        LEFT JOIN cameras c ON c.id = ar.camera_id
        WHERE 1=1
    """
    params = {}
    if date_filter == "today":
        sql += " AND ar.created_at >= CURRENT_DATE"
    sql += " ORDER BY ar.created_at DESC;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


@tool
def generate_safety_summary_report(
    camera_id_or_name: Optional[str] = None,
    date_filter: Optional[Literal["today", "this_week"]] = None
) -> Dict[str, Any]:
    """
    Aggregates safety parameters, alert metrics, and incident volume statistics to produce a safety report.
    """
    if engine is None: return {"error": "Database connector is uninitialized."}
    
    where_alerts = "1=1"
    where_incidents = "1=1"
    where_hse = "1=1"
    params = {}
    
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            where_alerts += " AND (a.camera_id = :cid OR a.camera_name ILIKE :cam_name_like)"
            where_incidents += " AND (i.camera_id = :cid OR i.camera_name ILIKE :cam_name_like)"
            where_hse += " AND (hre.camera_id = :cid OR c.name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            where_alerts += " AND a.camera_name ILIKE :cam_name_like"
            where_incidents += " AND i.camera_name ILIKE :cam_name_like"
            where_hse += " AND c.name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
            
    if date_filter == "today":
        where_alerts += " AND a.created_at >= CURRENT_DATE"
        where_incidents += " AND i.started_at >= CURRENT_DATE"
        where_hse += " AND hre.triggered_at >= CURRENT_DATE"
    elif date_filter == "this_week":
        where_alerts += " AND a.created_at >= DATE_TRUNC('week', CURRENT_DATE)"
        where_incidents += " AND i.started_at >= DATE_TRUNC('week', CURRENT_DATE)"
        where_hse += " AND hre.triggered_at >= DATE_TRUNC('week', CURRENT_DATE)"
        
    sql_alerts = f"SELECT COUNT(*) FROM alerts a WHERE {where_alerts}"
    sql_incidents = f"SELECT COUNT(*), COUNT(CASE WHEN is_active=true THEN 1 END) FROM incidents i WHERE {where_incidents}"
    sql_hse = f"SELECT COUNT(*) FROM hse_rule_events hre LEFT JOIN cameras c ON c.id = hre.camera_id WHERE {where_hse}"
    
    try:
        with engine.connect() as conn:
            alerts_count = conn.execute(text(sql_alerts), params).scalar()
            incidents_row = conn.execute(text(sql_incidents), params).fetchone()
            hse_count = conn.execute(text(sql_hse), params).scalar()
            
            total_inc = incidents_row[0] if incidents_row else 0
            active_inc = incidents_row[1] if incidents_row else 0
            
            return {
                "alerts_generated": alerts_count,
                "total_incidents": total_inc,
                "active_incidents": active_inc,
                "hse_rule_violations": hse_count,
                "summary_period": date_filter or "all_time",
                "filter_camera": camera_id_or_name or "all"
            }
    except Exception as e:
        return {"error": str(e)}


@tool
def get_event_timeline(
    camera_id_or_name: Optional[str] = None,
    start_time: Optional[str] = None,
    end_time: Optional[str] = None,
    date_filter: Optional[Literal["today"]] = None
) -> List[Dict[str, Any]]:
    """
    Retrieves a combined chronological timeline of surveillance events (alerts, incidents, and HSE violations).
    Can filter by time of day (e.g. start_time='10:00', end_time='12:00').
    """
    if engine is None: return [{"error": "Database connector is uninitialized."}]
    
    where_alerts = "1=1"
    where_incidents = "1=1"
    where_hse = "1=1"
    params = {}
    
    if camera_id_or_name is not None:
        try:
            cid = int(camera_id_or_name)
            where_alerts += " AND (a.camera_id = :cid OR a.camera_name ILIKE :cam_name_like)"
            where_incidents += " AND (i.camera_id = :cid OR i.camera_name ILIKE :cam_name_like)"
            where_hse += " AND (hre.camera_id = :cid OR c.name ILIKE :cam_name_like)"
            params["cid"] = cid
            params["cam_name_like"] = f"%{camera_id_or_name}%"
        except ValueError:
            where_alerts += " AND a.camera_name ILIKE :cam_name_like"
            where_incidents += " AND i.camera_name ILIKE :cam_name_like"
            where_hse += " AND c.name ILIKE :cam_name_like"
            params["cam_name_like"] = f"%{camera_id_or_name}%"
            
    if date_filter == "today":
        where_alerts += " AND a.created_at >= CURRENT_DATE"
        where_incidents += " AND i.started_at >= CURRENT_DATE"
        where_hse += " AND hre.triggered_at >= CURRENT_DATE"
        
    if start_time is not None:
        where_alerts += " AND a.created_at::time >= :start_time::time"
        where_incidents += " AND i.started_at::time >= :start_time::time"
        where_hse += " AND hre.triggered_at::time >= :start_time::time"
        params["start_time"] = start_time
    if end_time is not None:
        where_alerts += " AND a.created_at::time <= :end_time::time"
        where_incidents += " AND i.started_at::time <= :end_time::time"
        where_hse += " AND hre.triggered_at::time <= :end_time::time"
        params["end_time"] = end_time
        
    sql = f"""
        (SELECT 'alert' as event_type, a.id, a.created_at as timestamp, a.camera_name, a.class_name as details
         FROM alerts a WHERE {where_alerts})
        UNION ALL
        (SELECT 'incident' as event_type, i.id, i.started_at as timestamp, i.camera_name, i.class_name || ' (incident, active=' || i.is_active || ')' as details
         FROM incidents i WHERE {where_incidents})
        UNION ALL
        (SELECT 'hse_rule' as event_type, hre.id, hre.triggered_at as timestamp, c.name as camera_name, hrd.name || ' (severity=' || hre.severity || ')' as details
         FROM hse_rule_events hre
         LEFT JOIN cameras c ON c.id = hre.camera_id
         LEFT JOIN hse_rule_definitions hrd ON hrd.id = hre.rule_id
         WHERE {where_hse})
        ORDER BY timestamp DESC
        LIMIT 100;
    """
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), params)
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]


# Global assignment array explicitly tracking System Agent components


def _schema_lookup_helper(table_name: Optional[str] = None) -> List[Dict[str, Any]]:
    if engine is None: return [{"error": "Database uninitialized."}]
    if table_name:
        sql = """
            SELECT column_name, data_type, is_nullable, column_default
            FROM information_schema.columns
            WHERE table_schema = 'public' AND table_name = :table
            ORDER BY ordinal_position;
        """
        params = {"table": table_name}
    else:
        sql = """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' 
            ORDER BY table_name;
        """
        params = {}
    with engine.connect() as conn:
        result = conn.execute(text(sql), params)
        return _serialize_row_data(result)

@tool
def get_current_user() -> Dict[str, Any]:
    """
    Retrieves complete profile details of the currently authenticated user session.
    """
    uid = globals().get("user_id", 1)
    if engine is None: return {"error": "Database connector is uninitialized."}
    query = """
        SELECT u.id, u.full_name, u.email, u.username, r.name as role_name, u.is_active, u.created_at, d.name as department_name, des.name as designation_title, u.phone, u.employee_id
        FROM users u
        LEFT JOIN roles r ON r.id = u.role_id
        LEFT JOIN departments d ON d.id = u.department_id
        LEFT JOIN designations des ON des.id = u.designation_id
        WHERE u.id = :uid;
    """
    try:
        with engine.connect() as conn:
            row = conn.execute(text(query), {"uid": uid}).fetchone()
            if not row:
                return {"error": "User profile not found"}
            return _serialize_row_data([row])[0]
    except Exception as e:
        return {"error": str(e)}

@tool
def list_entities(entity: Literal["rules", "cameras", "zones", "models", "users", "notifications", "incidents"]) -> List[Dict[str, Any]]:
    """
    Lists all records for a specified entity type (rules, cameras, zones, models, users, notifications, incidents).
    """
    if engine is None: return [{"error": "Database uninitialized."}]
    entity_map = {
        "rules": "hse_rule_definitions",
        "cameras": "cameras",
        "zones": "zones",
        "models": "ai_models",
        "users": "users",
        "notifications": "notification_rules",
        "incidents": "incidents"
    }
    table = entity_map.get(entity)
    if not table:
        return [{"error": f"Unknown entity type: {entity}"}]
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT * FROM {table} ORDER BY id LIMIT 100;"))
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]

@tool
def count_entities(entity: Literal["rules", "cameras", "zones", "models", "users", "notifications", "incidents"]) -> Dict[str, Any]:
    """
    Counts total records for a specified entity type (rules, cameras, zones, models, users, notifications, incidents).
    """
    if engine is None: return {"error": "Database uninitialized."}
    entity_map = {
        "rules": "hse_rule_definitions",
        "cameras": "cameras",
        "zones": "zones",
        "models": "ai_models",
        "users": "users",
        "notifications": "notification_rules",
        "incidents": "incidents"
    }
    table = entity_map.get(entity)
    if not table:
        return {"error": f"Unknown entity type: {entity}"}
    try:
        with engine.connect() as conn:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {table};")).scalar()
            return {"entity": entity, "count": count}
    except Exception as e:
        return {"error": str(e)}

@tool
def search_rules(query: str) -> List[Dict[str, Any]]:
    """
    Searches HSE safety rules by matching a query string in rule name, description, condition_tree, or alert_config.
    """
    if engine is None: return [{"error": "Database uninitialized."}]
    sql = """
        SELECT * FROM hse_rule_definitions
        WHERE name ILIKE :q
           OR description ILIKE :q
           OR condition_tree::text ILIKE :q
           OR alert_config::text ILIKE :q;
    """
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), {"q": f"%{query}%"})
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]

@tool
def search_entities(entity: Literal["rules", "cameras", "zones", "models", "users", "notifications", "incidents"], query: str) -> List[Dict[str, Any]]:
    """
    Searches any entity type (rules, cameras, zones, models, users, notifications, incidents) by matching a text query.
    """
    if engine is None: return [{"error": "Database uninitialized."}]
    entity_map = {
        "rules": ("hse_rule_definitions", ["name", "description"]),
        "cameras": ("cameras", ["name", "ip", "camera_number"]),
        "zones": ("zones", ["name", "zone_type", "description"]),
        "models": ("ai_models", ["name", "framework", "description"]),
        "users": ("users", ["full_name", "email", "username"]),
        "notifications": ("notification_rules", ["name"]),
        "incidents": ("incidents", ["camera_name", "class_name", "root_cause"])
    }
    mapping = entity_map.get(entity)
    if not mapping:
        return [{"error": f"Unknown entity: {entity}"}]
    table, columns = mapping
    conditions = " OR ".join([f"{col} ILIKE :q" for col in columns])
    sql = f"SELECT * FROM {table} WHERE {conditions} LIMIT 100;"
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), {"q": f"%{query}%"})
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]

@tool
def get_entity_details(entity: Literal["rule", "camera", "zone", "model", "user", "notification", "incident"], id: int) -> Dict[str, Any]:
    """
    Retrieves complete record details for a specific entity type and primary key integer ID.
    """
    if engine is None: return {"error": "Database uninitialized."}
    entity_map = {
        "rule": "hse_rule_definitions",
        "camera": "cameras",
        "zone": "zones",
        "model": "ai_models",
        "user": "users",
        "notification": "notification_rules",
        "incident": "incidents"
    }
    table = entity_map.get(entity)
    if not table:
        return {"error": f"Unknown entity type: {entity}"}
    try:
        with engine.connect() as conn:
            row = conn.execute(text(f"SELECT * FROM {table} WHERE id = :id;"), {"id": id}).fetchone()
            if not row:
                return {"error": f"Record with ID {id} not found in {table}."}
            return _serialize_row_data([row])[0]
    except Exception as e:
        return {"error": str(e)}

@tool
def find_relationships(source: str, target: str, source_id: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Finds related records between source and target entities (e.g. Which cameras use Rule 16?, Which model is assigned to Camera 3?).
    """
    if engine is None: return [{"error": "Database uninitialized."}]
    src = source.lower().strip()
    tgt = target.lower().strip()
    
    if src == "rule" and tgt == "camera":
        sql = """
            SELECT c.* FROM cameras c
            JOIN hse_camera_rules hcr ON hcr.camera_id = c.id
            WHERE hcr.rule_id = :sid;
        """
    elif src == "camera" and tgt == "model":
        sql = """
            SELECT m.* FROM ai_models m
            JOIN detection_assignments da ON da.model_id = m.id
            WHERE da.camera_id = :sid;
        """
    elif src == "camera" and tgt == "zone":
        sql = "SELECT * FROM zones WHERE camera_id = :sid;"
    elif src == "camera" and tgt == "rule":
        sql = """
            SELECT hrd.* FROM hse_rule_definitions hrd
            JOIN hse_camera_rules hcr ON hcr.rule_id = hrd.id
            WHERE hcr.camera_id = :sid;
        """
    else:
        return [{"error": f"Relationship query from '{source}' to '{target}' is not supported directly."}]
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql), {"sid": source_id})
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]

@tool
def database_query(question: str, sql_query: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Runs a read-only SQL query against the system database to answer a natural language question. Only use this when no specialized tool can answer.
    """
    if not sql_query:
        return [{"error": "Please provide the corresponding SQL query in the 'sql_query' argument."}]
    return query_system_data.invoke({"sql_query": sql_query})

@tool
def schema_lookup(question: str) -> List[Dict[str, Any]]:
    """
    Looks up database schemas and table definitions to identify table names and fields (e.g. Which table stores HSE rules?).
    """
    q = question.lower()
    table = None
    if "hse_camera_rule" in q or "hse rule assignment" in q:
        table = "hse_camera_rules"
    elif "hse_rule_definition" in q or "hse rule" in q or "rule definitions" in q:
        table = "hse_rule_definitions"
    elif "camera_status" in q:
        table = "camera_status_logs"
    elif "camera" in q:
        table = "cameras"
    elif "zone" in q:
        table = "zones"
    elif "model" in q:
        table = "ai_models"
    elif "user" in q:
        table = "users"
    elif "incident" in q:
        table = "incidents"
    elif "alert" in q:
        table = "alerts"
    try:
        return _schema_lookup_helper(table)
    except Exception as e:
        return [{"error": str(e)}]

@tool
def aggregate_data(question: str) -> List[Dict[str, Any]]:
    """
    Performs statistics and data aggregations on the database (e.g. Top risky zones, Most active cameras, Most common incident, incident count this week).
    """
    if engine is None: return [{"error": "Database uninitialized."}]
    q = question.lower()
    sql = None
    if "risky" in q or "risk" in q:
        sql = """
            SELECT z.name as zone_name, zrs.risk_score, zrs.risk_level
            FROM zone_risk_scores zrs
            LEFT JOIN zones z ON z.id = zrs.zone_id
            ORDER BY zrs.risk_score DESC LIMIT 5;
        """
    elif "incident" in q:
        if "week" in q:
            sql = "SELECT COUNT(*) as weekly_incident_count FROM incidents WHERE started_at >= DATE_TRUNC('week', CURRENT_DATE);"
        else:
            sql = "SELECT class_name, COUNT(*) as occurrence_count FROM incidents GROUP BY class_name ORDER BY occurrence_count DESC LIMIT 5;"
    elif "camera" in q:
        sql = "SELECT camera_name, COUNT(*) as alert_count FROM alerts GROUP BY camera_name ORDER BY alert_count DESC LIMIT 5;"
    else:
        return [{"error": "Unsupported aggregate question."}]
    try:
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            return _serialize_row_data(result)
    except Exception as e:
        return [{"error": str(e)}]

system_agent_tools_registry = [
    query_system_data, 
    check_camera_fleet_health, 
    fetch_active_safety_alerts, 
    get_production_counting_summary,
    get_cameras,
    get_zones,
    get_incidents,
    get_alerts,
    get_anomalies,
    get_zone_risk_scores,
    get_hse_rule_violations,
    get_defect_detections,
    get_counting_statistics,
    get_recording_history,
    get_notification_history,
    get_scheduled_reports,
    get_system_settings,
    get_users_and_activity,
    get_ai_recommendations,
    generate_safety_summary_report,
    get_event_timeline,
    get_current_user,
    list_entities,
    count_entities,
    search_rules,
    search_entities,
    get_entity_details,
    find_relationships,
    database_query,
    schema_lookup,
    aggregate_data
]


# Incident Investigator Agent

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional, Literal

class ForensicWizardOutput(BaseModel):
    incident_id: List[int] = Field(description="List of matching alert, incident, or HSE event IDs.")
    summary: str = Field(description="A detailed summary of all safety and HSE activity for the complete day, strictly exactly 5 lines.")
    response: str = Field(description="A single-line response addressing the user's specific query based on matched events.")

class InvestigationIntent(BaseModel):
    intent_type: Literal["search", "deep"] = Field(
        description="Determine whether this is a broad retrieval search ('search') or an in-depth autopsy investigation query ('deep')."
    )
    date_str: Optional[str] = Field(
        None,
        description="The target date of the query formatted strictly as YYYY-MM-DD. Resolve relative dates like 'today', 'yesterday' based on current system time."
    )
    camera_filter: Optional[str] = Field(
        None,
        description="Any camera name, number, or ID mentioned in the query."
    )
    zone_filter: Optional[str] = Field(
        None,
        description="Any zone name or type mentioned in the query."
    )
    class_name: Optional[str] = Field(
        None,
        description="Any specific event/class name mentioned (e.g., helmet, fire, ppe, smoke, person, forklift)."
    )

@tool
def investigate_events(user_query: str) -> Dict[str, Any]:
    """
    Driven 100% by natural language. Investigate safety events, alarms, incidents, and HSE violations.
    The tool internally extracts search or deep intent, target dates, camera and zone constraints,
    queries the database, and returns either a broad chronological summary or a deep autopsy analysis.
    """
    if engine is None: 
        return {"error": "Database connector is uninitialized."}
        
    from datetime import datetime
    current_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # 1. Resolve relative dates/extract intent using base_llm
    intent_prompt = f"""
    Analyze the user query: "{user_query}"
    CURRENT SYSTEM DATE AND TIME: {current_time_str}.
    Resolve relative dates like 'today' or 'yesterday' to absolute dates in YYYY-MM-DD format.
    Extract the investigation intent parameters.
    """
    
    try:
        structured_llm = base_llm.with_structured_output(InvestigationIntent)
        intent = structured_llm.invoke(intent_prompt)
    except Exception as e:
        intent = InvestigationIntent(
            intent_type="search",
            date_str=datetime.now().strftime("%Y-%m-%d")
        )
        
    date_str = intent.date_str or datetime.now().strftime("%Y-%m-%d")
    camera_filter = intent.camera_filter
    zone_filter = intent.zone_filter
    class_filter = intent.class_name
    
    # helper to resolve camera and zone
    cam_id = None
    zone_id = None
    try:
        with engine.connect() as conn:
            if camera_filter:
                cam_id = _resolve_camera_id(conn, camera_filter)
            if zone_filter and cam_id:
                zone_id = _resolve_zone_id(conn, cam_id, zone_filter)
    except Exception:
        pass

    # Helper function for search execution
    def execute_search_queries(date_val, c_id, z_id, cl_filter):
        try:
            with engine.connect() as conn:
                # Build Alerts query
                alerts_query = "SELECT a.id, a.class_name, a.snapshot_path, a.created_at::time AS event_time FROM alerts a LEFT JOIN zones z ON z.id = a.zone_id WHERE a.created_at::date = :date_val"
                params = {"date_val": date_val}
                if c_id:
                    alerts_query += " AND a.camera_id = :cam_id"
                    params["cam_id"] = c_id
                if z_id:
                    alerts_query += " AND a.zone_id = :zone_id"
                    params["zone_id"] = z_id
                if cl_filter:
                    alerts_query += " AND a.class_name ILIKE :class_filter"
                    params["class_filter"] = f"%{cl_filter}%"
                alerts_query += " ORDER BY event_time ASC;"
                alerts = _serialize_row_data(conn.execute(text(alerts_query), params).fetchall())
                
                # Build HSE query
                hse_query = "SELECT hre.id, hrd.name AS rule_name, hre.snapshot_path, hre.triggered_at::time AS event_time FROM hse_rule_events hre LEFT JOIN hse_rule_definitions hrd ON hrd.id = hre.rule_id LEFT JOIN zones z ON z.id = hre.zone_id WHERE hre.triggered_at::date = :date_val"
                params_hse = {"date_val": date_val}
                if c_id:
                    hse_query += " AND hre.camera_id = :cam_id"
                    params_hse["cam_id"] = c_id
                if z_id:
                    hse_query += " AND hre.zone_id = :zone_id"
                    params_hse["zone_id"] = z_id
                if cl_filter:
                    hse_query += " AND (hrd.name ILIKE :class_filter OR hrd.description ILIKE :class_filter)"
                    params_hse["class_filter"] = f"%{cl_filter}%"
                hse_query += " ORDER BY event_time ASC;"
                hse = _serialize_row_data(conn.execute(text(hse_query), params_hse).fetchall())
                
                # Build Incidents query
                incidents_query = "SELECT i.id, i.class_name, i.started_at::time AS event_time, i.root_cause, i.is_active FROM incidents i LEFT JOIN zones z ON z.id = i.zone_id WHERE i.started_at::date = :date_val"
                params_inc = {"date_val": date_val}
                if c_id:
                    incidents_query += " AND i.camera_id = :cam_id"
                    params_inc["cam_id"] = c_id
                if z_id:
                    incidents_query += " AND i.zone_id = :zone_id"
                    params_inc["zone_id"] = z_id
                if cl_filter:
                    incidents_query += " AND i.class_name ILIKE :class_filter"
                    params_inc["class_filter"] = f"%{cl_filter}%"
                incidents_query += " ORDER BY event_time ASC;"
                incidents = _serialize_row_data(conn.execute(text(incidents_query), params_inc).fetchall())
                
                return alerts, hse, incidents
        except Exception as e:
            return [], [], []

    # 2. Broad Search vs Deep Autopsy Decision
    if intent.intent_type == "search":
        alerts, hse, incidents = execute_search_queries(date_str, cam_id, zone_id, class_filter)
        return {
            "investigation_type": "search",
            "date": date_str,
            "alerts": alerts,
            "hse_rule_events": hse,
            "incidents": incidents
        }
    else:
        # Deep Investigation Mode
        # A. Find the most relevant incident internally
        target_incident_id = None
        try:
            with engine.connect() as conn:
                find_sql = "SELECT id, class_name, root_cause, started_at FROM incidents WHERE started_at::date = :date_val"
                params = {"date_val": date_str}
                if cam_id:
                    find_sql += " AND camera_id = :cam_id"
                    params["cam_id"] = cam_id
                if zone_id:
                    find_sql += " AND zone_id = :zone_id"
                    params["zone_id"] = zone_id
                if class_filter:
                    find_sql += " AND class_name ILIKE :class_filter"
                    params["class_filter"] = f"%{class_filter}%"
                find_sql += " ORDER BY started_at DESC;"
                
                incidents_found = conn.execute(text(find_sql), params).fetchall()
                if incidents_found:
                    # Look for critical incidents first (e.g. fire/smoke) or default to the most recent one
                    best_match = incidents_found[0]
                    for row in incidents_found:
                        cls = str(row._mapping.get("class_name", "")).lower()
                        if "fire" in cls or "smoke" in cls:
                            best_match = row
                            break
                    target_incident_id = best_match._mapping["id"]
        except Exception:
            pass
            
        if not target_incident_id:
            # Fall back to broad search
            alerts, hse, incidents = execute_search_queries(date_str, cam_id, zone_id, class_filter)
            return {
                "investigation_type": "search_fallback",
                "message": "No specific matching incident found for deep autopsy. Displaying broad safety events instead.",
                "date": date_str,
                "alerts": alerts,
                "hse_rule_events": hse,
                "incidents": incidents
            }
            
        # B. Run autopsy logic internally for the target_incident_id
        try:
            with engine.connect() as conn:
                inc_sql = text("""
                    SELECT id, camera_id, camera_name, zone_id, class_name, confidence, started_at, resolved_at, duration_seconds, is_active, root_cause
                    FROM incidents
                    WHERE id = :id;
                """)
                inc_row = conn.execute(inc_sql, {"id": target_incident_id}).fetchone()
                if not inc_row:
                    return {"error": f"Incident with ID {target_incident_id} not found."}
                    
                incident_data = dict(inc_row._mapping)
                started_at = incident_data["started_at"]
                camera_id = incident_data["camera_id"]
                zone_id = incident_data["zone_id"]
                class_name = incident_data["class_name"]
                
                # Serialize datetimes
                incident_data["started_at"] = str(started_at)
                if incident_data.get("resolved_at"):
                    incident_data["resolved_at"] = str(incident_data["resolved_at"])
                    
                from datetime import timedelta
                start_win = started_at - timedelta(minutes=10)
                end_win = started_at + timedelta(minutes=10)
                
                timeline_incidents_sql = text("""
                    SELECT id, class_name, started_at, resolved_at, is_active, root_cause
                    FROM incidents
                    WHERE (camera_id = :cam_id OR zone_id = :zone_id)
                      AND id != :inc_id
                      AND started_at BETWEEN :start_win AND :end_win
                    ORDER BY started_at ASC;
                """)
                
                timeline_alerts_sql = text("""
                    SELECT id, class_name, confidence, created_at
                    FROM alerts
                    WHERE (camera_id = :cam_id OR zone_id = :zone_id)
                      AND created_at BETWEEN :start_win AND :end_win
                    ORDER BY created_at ASC;
                """)
                
                inc_rows = conn.execute(timeline_incidents_sql, {
                    "cam_id": camera_id,
                    "zone_id": zone_id,
                    "inc_id": target_incident_id,
                    "start_win": start_win,
                    "end_win": end_win
                }).fetchall()
                
                alert_rows = conn.execute(timeline_alerts_sql, {
                    "cam_id": camera_id,
                    "zone_id": zone_id,
                    "start_win": start_win,
                    "end_win": end_win
                }).fetchall()
                
                timeline_incidents = _serialize_row_data(inc_rows)
                for item in timeline_incidents:
                    item["started_at"] = str(item["started_at"])
                    if item.get("resolved_at"):
                        item["resolved_at"] = str(item["resolved_at"])
                        
                timeline_alerts = _serialize_row_data(alert_rows)
                for item in timeline_alerts:
                    item["created_at"] = str(item["created_at"])
                    
                history_sql = text("""
                    SELECT id, started_at, resolved_at, duration_seconds, root_cause
                    FROM incidents
                    WHERE zone_id = :zone_id
                      AND class_name = :class_name
                      AND id != :inc_id
                    ORDER BY started_at DESC
                    LIMIT 5;
                """)
                hist_rows = conn.execute(history_sql, {
                    "zone_id": zone_id,
                    "class_name": class_name,
                    "inc_id": target_incident_id
                }).fetchall()
                
                history = _serialize_row_data(hist_rows)
                for item in history:
                    item["started_at"] = str(item["started_at"])
                    if item.get("resolved_at"):
                        item["resolved_at"] = str(item["resolved_at"])
                        
                return {
                    "investigation_type": "deep",
                    "incident_id": target_incident_id,
                    "incident": incident_data,
                    "timeline": {
                        "incidents": timeline_incidents,
                        "alerts": timeline_alerts
                    },
                    "history": history
                }
        except Exception as e:
            return {"error": f"Failed to generate autopsy: {str(e)}"}

investigator_agent_tools_registry = [investigate_events]


# Setup Agent

In [ ]:
from sqlalchemy import create_engine, text
from langchain_core.tools import tool
from typing import List, Dict, Any, Optional, Literal
import json
# ====================================================================
# 🛡️ DYNAMIC ROLE-BASED ACCESS CONTROL (RBAC) LAYER
# ====================================================================
TOOL_TO_COMPONENT_MAP = {
    # Cameras
    "get_video_stream_url": "Live Streaming",
    "analyze_video_feed": "Live Streaming",
    "analyze_scene_context": "Live Streaming",
    "semantic_search_scene_history": "Live Streaming",
    "check_camera_fleet_health": "Cameras",
    "get_cameras": "Cameras",
    "manage_camera": "Cameras",
    "get_recording_history": "Cameras",
    "get_live_stream_health": "Live Streaming",
    "capture_live_snapshot": "Live Streaming",
    "get_live_people_count_multi_camera": "Live Streaming",
    "detect_motion_in_stream": "Live Streaming",
    "find_person_by_description_live": "Live Streaming",
    "get_live_ppe_compliance_check": "Live Streaming",
    
    # Zones
    "get_zones": "Zones",
    "create_or_update_zone": "Zones",
    "delete_zone_by_name": "Zones",
    "get_zone_risk_scores": "Zones",
    
    # Incidents
    "get_incidents": "Incidents",
    "get_hse_rule_violations": "Incidents",
    "get_defect_detections": "Incidents",
    "investigate_events": "Incidents",
    
    
    # Alerts
    "fetch_active_safety_alerts": "Alerts",
    "get_alerts": "Alerts",
    "get_anomalies": "Alerts",
    "get_notification_history": "Alerts",
    
    # Assignments
    "create_or_update_detection_assignment": "Assignments",
    "create_or_update_hse_rule": "Assignments",
    "delete_rule_by_name": "Assignments",
    "manage_recipient_group": "Assignments",
    "manage_recipient": "Assignments",
    "create_or_update_notification_rule": "Assignments",
    "update_notification_settings": "Assignments",
    "manage_scheduled_report": "Assignments",
    "manage_user": "Assignments",
    "manage_ai_model": "Assignments",
    "acknowledge_telemetry_flags": "Assignments",
    
    # Dashboard
    "get_production_counting_summary": "Dashboard",
    "get_event_timeline": "Dashboard",
    "generate_safety_summary_report": "Dashboard",
    "get_counting_statistics": "Dashboard",
    "get_scheduled_reports": "Dashboard",
    "get_system_settings": "Dashboard",
    "get_users_and_activity": "Dashboard",
    "get_ai_recommendations": "Dashboard",
    
    # Profile
    "get_current_user_profile": "Dashboard",
    
    # Generic Data Utilities
    "get_current_user": "Dashboard",
    "list_entities": "Dashboard",
    "count_entities": "Dashboard",
    "search_rules": "Dashboard",
    "search_entities": "Dashboard",
    "get_entity_details": "Dashboard",
    "find_relationships": "Dashboard",
    "database_query": "Dashboard",
    "schema_lookup": "Dashboard",
    "aggregate_data": "Dashboard"
}
def get_user_permissions(user_id_or_username: str) -> List[str]:
    """Retrieves allowed components dynamically from database schema."""
    if engine is None:
        return []
    try:
        with engine.connect() as conn:
            user_sql = """
                SELECT u.role_id, r.name as role_name 
                FROM users u
                LEFT JOIN roles r ON r.id = u.role_id
                WHERE u.id = :id OR u.username = :uname OR u.email = :email
            """
            params = {
                "id": None,
                "uname": str(user_id_or_username),
                "email": str(user_id_or_username)
            }
            try:
                params["id"] = int(user_id_or_username)
            except ValueError:
                pass
                
            user_row = conn.execute(text(user_sql), params).fetchone()
            if not user_row:
                return []
            role_id, role_name = user_row
            
            # If Administrator, dynamically grant access to all components
            if role_name and role_name.lower() in ["administrator", "admin"]:
                comps = conn.execute(text("SELECT DISTINCT component FROM role_permissions")).fetchall()
                allowed = [c[0] for c in comps if c[0]]
                core = ["Dashboard", "Cameras", "Assignments", "Live Streaming", "Zones", "Incidents", "Alerts"]
                for c in core:
                    if c not in allowed:
                        allowed.append(c)
                return allowed
                
            perm_sql = "SELECT DISTINCT component FROM role_permissions WHERE role_id = :role_id"
            rows = conn.execute(text(perm_sql), {"role_id": role_id}).fetchall()
            return [r[0] for r in rows if r[0]]
    except Exception as e:
        print(f"Error fetching permissions: {e}")
        return []
def get_required_component_for_query(query: str) -> Optional[str]:
    """Maps operator intent keywords to necessary RBAC module."""
    q = query.lower()
    if any(k in q for k in ["video stream", "rtsp feed", "live stream", "video feed", "camera stream", "count person", "count people", "detect", "worker doing", "what is happening", "anyone in"]):
        return "Live Streaming"
    if any(k in q for k in ["camera"]):
        return "Cameras"
    if any(k in q for k in ["zone"]):
        return "Zones"
    if any(k in q for k in ["incident", "hse event", "violation"]):
        return "Incidents"
    if any(k in q for k in ["alert", "anomaly"]):
        return "Alerts"
    if any(k in q for k in ["assignment", "rule"]):
        return "Assignments"
    if any(k in q for k in ["counting", "recordings", "snapshots"]):
        return "Dashboard"
    if any(k in q for k in ["notification", "settings", "recipient", "smtp", "bot token"]):
        return "Assignments"
    if any(k in q for k in ["scheduled report"]):
        return "Dashboard"
    if any(k in q for k in ["user", "profile"]):
        return "Dashboard"
    if any(k in q for k in ["ai model", "model"]):
        return "Assignments"
    return None
# ====================================================================
# 🛡️ HELPER RESOLUTION FUNCTIONS FOR SETUP MUTATIONS
# ====================================================================
def _resolve_camera_id(conn, camera_name_or_id: str) -> Optional[int]:
    try:
        cid = int(camera_name_or_id)
        res = conn.execute(text("SELECT id FROM cameras WHERE id = :id"), {"id": cid}).fetchone()
        if res: return res[0]
    except ValueError:
        pass
    res = conn.execute(text("SELECT id FROM cameras WHERE name ILIKE :val OR camera_number = :val_str"),
                       {"val": camera_name_or_id, "val_str": camera_name_or_id}).fetchone()
    if res: return res[0]
    return None
def _resolve_zone_id(conn, camera_id: int, zone_name: str) -> Optional[int]:
    res = conn.execute(text("SELECT id FROM zones WHERE camera_id = :cam_id AND (name ILIKE :val OR name = :exact)"),
                       {"cam_id": camera_id, "val": f"%{zone_name}%", "exact": zone_name}).fetchone()
    if res: return res[0]
    return None
def _resolve_model_id(conn, model_name_or_id: str) -> Optional[int]:
    try:
        mid = int(model_name_or_id)
        res = conn.execute(text("SELECT id FROM ai_models WHERE id = :id"), {"id": mid}).fetchone()
        if res: return res[0]
    except ValueError:
        pass
    res = conn.execute(text("SELECT id FROM ai_models WHERE name ILIKE :val"), {"val": model_name_or_id}).fetchone()
    if res: return res[0]
    return None
def _resolve_class_id(conn, model_id: int, class_name: str) -> Optional[int]:
    res = conn.execute(text("SELECT id FROM ai_model_classes WHERE model_id = :mid AND class_name ILIKE :val"),
                       {"mid": model_id, "val": class_name}).fetchone()
    if res: return res[0]
    return None
def _resolve_rule_id(conn, rule_name: str) -> Optional[int]:
    res = conn.execute(text("SELECT id FROM hse_rule_definitions WHERE name ILIKE :val"), {"val": rule_name}).fetchone()
    if res: return res[0]
    res = conn.execute(text("SELECT id FROM hse_rule_definitions WHERE name ILIKE :val"), {"val": f"%{rule_name}%"}).fetchone()
    if res: return res[0]
    return None
# ====================================================================
# 🛠️ SETUP AGENT WRITE/UPDATE MUTATION TOOLS
# ====================================================================
@tool
def manage_recipient_group(group_name: str, action: Literal["create", "delete"], description: Optional[str] = None) -> Dict[str, Any]:
    """
    Manages recipient groups for alert notifications.
    Required fields for creation: group_name.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not group_name:
        return {"success": False, "error": "Missing required field: group_name. Please ask the operator for the recipient group name."}
        
    try:
        with engine.begin() as conn:
            if action == "create":
                res = conn.execute(text("SELECT id FROM recipient_groups WHERE name = :name"), {"name": group_name}).fetchone()
                if res:
                    if description is not None:
                        conn.execute(text("UPDATE recipient_groups SET description = :desc WHERE id = :id"), {"desc": description, "id": res[0]})
                    return {"success": True, "message": f"Recipient group '{group_name}' already exists. Updated description."}
                else:
                    conn.execute(text("INSERT INTO recipient_groups (name, description) VALUES (:name, :desc)"), {"name": group_name, "desc": description or ""})
                    return {"success": True, "message": f"Successfully created recipient group '{group_name}'."}
            elif action == "delete":
                res = conn.execute(text("SELECT id FROM recipient_groups WHERE name = :name"), {"name": group_name}).fetchone()
                if not res:
                    return {"success": False, "error": f"Recipient group '{group_name}' not found."}
                gid = res[0]
                conn.execute(text("DELETE FROM recipients WHERE group_id = :gid"), {"gid": gid})
                conn.execute(text("DELETE FROM notification_rule_recipients WHERE group_id = :gid"), {"gid": gid})
                conn.execute(text("DELETE FROM recipient_groups WHERE id = :gid"), {"gid": gid})
                return {"success": True, "message": f"Successfully deleted recipient group '{group_name}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage recipient group: {str(e)}"}
@tool
def manage_recipient(group_name: str, recipient: str, channel: Literal["email", "telegram", "whatsapp", "teams"], action: Literal["add", "remove"]) -> Dict[str, Any]:
    """
    Adds or removes a recipient contact information from a group.
    Required fields: group_name, recipient, channel.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not group_name:
        return {"success": False, "error": "Missing required field: group_name. Please ask the operator for the recipient group name."}
    if not recipient:
        return {"success": False, "error": "Missing required field: recipient. Please ask the operator for the recipient email, phone, or ID."}
    if not channel:
        return {"success": False, "error": "Missing required field: channel. Please ask the operator for the notification channel (email, telegram, whatsapp, teams)."}
        
    try:
        with engine.begin() as conn:
            res = conn.execute(text("SELECT id FROM recipient_groups WHERE name = :name"), {"name": group_name}).fetchone()
            if not res:
                return {"success": False, "error": f"Recipient group '{group_name}' does not exist. Please ask the operator if they want to create it first."}
            gid = res[0]
            
            if action == "add":
                exists = conn.execute(text("SELECT id FROM recipients WHERE group_id = :gid AND recipient = :rec AND channel = :chan"),
                                      {"gid": gid, "rec": recipient, "chan": channel}).fetchone()
                if exists:
                    return {"success": True, "message": f"Recipient '{recipient}' already in group '{group_name}' for channel '{channel}'."}
                conn.execute(text("INSERT INTO recipients (group_id, channel, recipient) VALUES (:gid, :chan, :rec)"),
                             {"gid": gid, "chan": channel, "rec": recipient})
                return {"success": True, "message": f"Successfully added '{recipient}' to group '{group_name}' on channel '{channel}'."}
            elif action == "remove":
                conn.execute(text("DELETE FROM recipients WHERE group_id = :gid AND recipient = :rec AND channel = :chan"),
                             {"gid": gid, "rec": recipient, "chan": channel})
                return {"success": True, "message": f"Successfully removed '{recipient}' from group '{group_name}' on channel '{channel}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage recipient: {str(e)}"}
@tool
def create_or_update_zone(camera_name_or_id: str, zone_name: str, zone_type: Optional[str] = None, coordinates: Optional[List[Any]] = None, is_active: Optional[bool] = None) -> Dict[str, Any]:
    """
    Creates or updates a zone configuration for a camera.
    Required fields for new zones: camera_name_or_id, zone_name, zone_type, coordinates.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not camera_name_or_id:
        return {"success": False, "error": "Missing required field: camera_name_or_id. Please ask the operator for the camera name or ID."}
    if not zone_name:
        return {"success": False, "error": "Missing required field: zone_name. Please ask the operator for the zone name."}
        
    try:
        # Normalize coordinates to [{"x": x, "y": y}, ...] to match database format
        normalized_coords = None
        if coordinates is not None:
            normalized_coords = []
            for item in coordinates:
                if isinstance(item, dict) and "x" in item and "y" in item:
                    normalized_coords.append({"x": float(item["x"]), "y": float(item["y"])})
                elif (isinstance(item, list) or isinstance(item, tuple)) and len(item) >= 2:
                    normalized_coords.append({"x": float(item[0]), "y": float(item[1])})
                    
        with engine.begin() as conn:
            cam_id = _resolve_camera_id(conn, camera_name_or_id)
            if not cam_id:
                return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
                
            res = conn.execute(text("SELECT id, zone_type, coordinates, is_active FROM zones WHERE camera_id = :cam_id AND name = :name"),
                               {"cam_id": cam_id, "name": zone_name}).fetchone()
            
            if res:
                zone_id = res[0]
                updates = {}
                if zone_type is not None: updates["zone_type"] = zone_type
                if coordinates is not None: updates["coordinates"] = json.dumps(normalized_coords)
                if is_active is not None: updates["is_active"] = is_active
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["id"] = zone_id
                    conn.execute(text(f"UPDATE zones SET {set_clause} WHERE id = :id"), updates)
                
                res_zone = conn.execute(text("SELECT id, camera_id, name, zone_type, coordinates, is_active FROM zones WHERE id = :id"), {"id": zone_id}).fetchone()
                record_data = dict(res_zone._mapping) if res_zone else {}
                if "coordinates" in record_data and isinstance(record_data["coordinates"], str):
                    try:
                        record_data["coordinates"] = json.loads(record_data["coordinates"])
                    except Exception:
                        pass
                return {"success": True, "message": f"Successfully updated zone '{zone_name}' on camera '{camera_name_or_id}'.", "record": record_data}
            else:
                if not zone_type:
                    return {"success": False, "error": "Missing required field: zone_type. Please ask the operator for the zone type (e.g. Red Zone, restricted)."}
                if not coordinates:
                    return {"success": False, "error": "Missing required field: coordinates. Please ask the operator for the coordinates list of the zone."}
                
                conn.execute(text("INSERT INTO zones (camera_id, name, zone_type, coordinates, is_active) VALUES (:cam_id, :name, :type, :coords, :active)"),
                             {"cam_id": cam_id, "name": zone_name, "type": zone_type, "coords": json.dumps(normalized_coords), "active": is_active if is_active is not None else True})
                
                res_zone = conn.execute(text("SELECT id, camera_id, name, zone_type, coordinates, is_active FROM zones WHERE camera_id = :cam_id AND name = :name ORDER BY id DESC LIMIT 1"),
                                        {"cam_id": cam_id, "name": zone_name}).fetchone()
                record_data = dict(res_zone._mapping) if res_zone else {}
                if "coordinates" in record_data and isinstance(record_data["coordinates"], str):
                    try:
                        record_data["coordinates"] = json.loads(record_data["coordinates"])
                    except Exception:
                        pass
                return {"success": True, "message": f"Successfully created new zone '{zone_name}' on camera '{camera_name_or_id}'.", "record": record_data}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage zone: {str(e)}"}
@tool
def delete_zone_by_name(camera_name_or_id: str, zone_name: str) -> Dict[str, Any]:
    """
    Deletes a zone by its name and camera mapping.
    Required fields: camera_name_or_id, zone_name.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not camera_name_or_id:
        return {"success": False, "error": "Missing required field: camera_name_or_id. Please ask the operator for the camera name or ID."}
    if not zone_name:
        return {"success": False, "error": "Missing required field: zone_name. Please ask the operator for the zone name."}
        
    try:
        with engine.begin() as conn:
            cam_id = _resolve_camera_id(conn, camera_name_or_id)
            if not cam_id:
                return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
            res = conn.execute(text("SELECT id FROM zones WHERE camera_id = :cam_id AND name = :name"),
                               {"cam_id": cam_id, "name": zone_name}).fetchone()
            if not res:
                return {"success": False, "error": f"Zone '{zone_name}' not found on camera '{camera_name_or_id}'."}
            zone_id = res[0]
            conn.execute(text("DELETE FROM hse_camera_rules WHERE zone_id = :zid"), {"zid": zone_id})
            conn.execute(text("DELETE FROM zones WHERE id = :zid"), {"zid": zone_id})
            return {"success": True, "message": f"Successfully deleted zone '{zone_name}' on camera '{camera_name_or_id}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to delete zone: {str(e)}"}
@tool
def create_or_update_detection_assignment(
    camera_name_or_id: str,
    model_name_or_id: str,
    class_name: Optional[str] = None,
    confidence_threshold: Optional[float] = None,
    alert_enabled: Optional[bool] = None,
    is_active: Optional[bool] = None
) -> Dict[str, Any]:
    """
    Assigns an AI model class detection task to a camera.
    Required fields for creation: camera_name_or_id, model_name_or_id.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not camera_name_or_id:
        return {"success": False, "error": "Missing required field: camera_name_or_id. Please ask the operator for the camera name or ID."}
    if not model_name_or_id:
        return {"success": False, "error": "Missing required field: model_name_or_id. Please ask the operator for the AI model name or ID."}
        
    try:
        with engine.begin() as conn:
            cam_id = _resolve_camera_id(conn, camera_name_or_id)
            if not cam_id:
                return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
            mod_id = _resolve_model_id(conn, model_name_or_id)
            if not mod_id:
                return {"success": False, "error": f"AI model '{model_name_or_id}' not found."}
                
            class_id = None
            if class_name:
                class_id = _resolve_class_id(conn, mod_id, class_name)
                if not class_id:
                    return {"success": False, "error": f"Class '{class_name}' not found for model '{model_name_or_id}'."}
            
            sql = "SELECT id FROM detection_assignments WHERE camera_id = :cam_id AND model_id = :mod_id"
            params = {"cam_id": cam_id, "mod_id": mod_id}
            if class_id is not None:
                sql += " AND class_id = :cls_id"
                params["cls_id"] = class_id
            else:
                sql += " AND class_id IS NULL"
                
            res = conn.execute(text(sql), params).fetchone()
            
            if res:
                da_id = res[0]
                updates = {}
                if confidence_threshold is not None: updates["confidence_threshold"] = confidence_threshold
                if alert_enabled is not None: updates["alert_enabled"] = alert_enabled
                if is_active is not None: updates["is_active"] = is_active
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["id"] = da_id
                    conn.execute(text(f"UPDATE detection_assignments SET {set_clause} WHERE id = :id"), updates)
                return {"success": True, "message": f"Successfully updated detection assignment for model '{model_name_or_id}' on camera '{camera_name_or_id}'."}
            else:
                conn.execute(text("""
                    INSERT INTO detection_assignments (camera_id, model_id, class_id, confidence_threshold, alert_enabled, is_active)
                    VALUES (:cam_id, :mod_id, :cls_id, :conf, :alert, :active)
                """), {
                    "cam_id": cam_id,
                    "mod_id": mod_id,
                    "cls_id": class_id,
                    "conf": confidence_threshold if confidence_threshold is not None else 0.5,
                    "alert": alert_enabled if alert_enabled is not None else True,
                    "active": is_active if is_active is not None else True
                })
                return {"success": True, "message": f"Successfully created new detection assignment for model '{model_name_or_id}' on camera '{camera_name_or_id}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage detection assignment: {str(e)}"}
@tool
def create_or_update_hse_rule(
    rule_name: str,
    camera_name_or_id: str,
    zone_name: Optional[str] = None,
    is_active: Optional[bool] = True,
    severity: Optional[str] = "high",
    config_override: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:
    """
    Creates or updates an HSE Safety Rule assignment (such as Red Zone intrusion, crowd density, loitering) for a camera.
    Required fields for creation: rule_name, camera_name_or_id.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not rule_name:
        return {"success": False, "error": "Missing required field: rule_name. Please ask the operator for the HSE rule name (e.g. Person Detection, Restricted Area Entry)."}
    if not camera_name_or_id:
        return {"success": False, "error": "Missing required field: camera_name_or_id. Please ask the operator for the camera name or ID."}
        
    try:
        with engine.begin() as conn:
            rule_id = _resolve_rule_id(conn, rule_name)
            if not rule_id:
                return {"success": False, "error": f"HSE rule definition '{rule_name}' not found."}
                
            cam_id = _resolve_camera_id(conn, camera_name_or_id)
            if not cam_id:
                return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
                
            zone_id = None
            if zone_name:
                zone_id = _resolve_zone_id(conn, cam_id, zone_name)
                if not zone_id:
                    return {"success": False, "error": f"Zone '{zone_name}' not found on camera '{camera_name_or_id}'."}
                    
            sql = "SELECT id FROM hse_camera_rules WHERE camera_id = :cam_id AND rule_id = :rule_id"
            params = {"cam_id": cam_id, "rule_id": rule_id}
            if zone_id is not None:
                sql += " AND zone_id = :zid"
                params["zid"] = zone_id
            else:
                sql += " AND zone_id IS NULL"
                
            res = conn.execute(text(sql), params).fetchone()
            
            if res:
                hcr_id = res[0]
                updates = {}
                if is_active is not None: updates["is_active"] = is_active
                if config_override is not None: updates["config_override"] = json.dumps(config_override)
                if severity is not None:
                    if config_override is None: config_override = {}
                    config_override["severity"] = severity
                    updates["config_override"] = json.dumps(config_override)
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["id"] = hcr_id
                    conn.execute(text(f"UPDATE hse_camera_rules SET {set_clause} WHERE id = :id"), updates)
                return {"success": True, "message": f"Successfully updated HSE rule '{rule_name}' on camera '{camera_name_or_id}'."}
            else:
                cfg = config_override or {}
                if severity:
                    cfg["severity"] = severity
                    
                conn.execute(text("""
                    INSERT INTO hse_camera_rules (camera_id, rule_id, zone_id, config_override, is_active)
                    VALUES (:cam_id, :rule_id, :zid, :cfg, :active)
                """), {
                    "cam_id": cam_id,
                    "rule_id": rule_id,
                    "zid": zone_id,
                    "cfg": json.dumps(cfg),
                    "active": is_active if is_active is not None else True
                })
                return {"success": True, "message": f"Successfully configured HSE rule '{rule_name}' on camera '{camera_name_or_id}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to assign HSE rule: {str(e)}"}
@tool
def create_or_update_notification_rule(
    rule_name: str,
    channels: List[str],
    camera_name_or_id: Optional[str] = None,
    zone_name: Optional[str] = None,
    class_name: Optional[str] = None,
    cooldown_seconds: Optional[int] = None,
    enabled: Optional[bool] = True,
    recipient_emails: Optional[List[str]] = None,
    recipient_group_name: Optional[str] = None
) -> Dict[str, Any]:
    """
    Creates or updates a notification rule to send alerts via Email, Telegram, Teams, etc.
    Required fields for creation: rule_name, channels.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not rule_name:
        return {"success": False, "error": "Missing required field: rule_name. Please ask the operator for the rule name."}
    if not channels:
        return {"success": False, "error": "Missing required field: channels. Please ask the operator which channels to use (e.g. Email, Telegram, Whatsapp)."}
        
    try:
        with engine.begin() as conn:
            res = conn.execute(text("SELECT id FROM notification_rules WHERE name = :name"), {"name": rule_name}).fetchone()
            
            rule_id = None
            if res:
                rule_id = res[0]
                updates = {"channels": json.dumps(channels)}
                if cooldown_seconds is not None: updates["cooldown_seconds"] = cooldown_seconds
                if enabled is not None: updates["enabled"] = enabled
                set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                updates["id"] = rule_id
                conn.execute(text(f"UPDATE notification_rules SET {set_clause} WHERE id = :id"), updates)
            else:
                rule_id = conn.execute(text("""
                    INSERT INTO notification_rules (name, channels, cooldown_seconds, enabled, attach_snapshot)
                    VALUES (:name, :chan, :cool, :enabled, true) RETURNING id
                """), {
                    "name": rule_name,
                    "chan": json.dumps(channels),
                    "cool": cooldown_seconds if cooldown_seconds is not None else 60,
                    "enabled": enabled
                }).scalar()
                
            cam_id = None
            if camera_name_or_id:
                cam_id = _resolve_camera_id(conn, camera_name_or_id)
            zone_id = None
            if zone_name and cam_id:
                zone_id = _resolve_zone_id(conn, cam_id, zone_name)
                
            class_id = None
            if class_name:
                c_res = conn.execute(text("SELECT id FROM ai_model_classes WHERE class_name ILIKE :name LIMIT 1"), {"name": class_name}).fetchone()
                if c_res:
                    class_id = c_res[0]
                    
            target_exists = conn.execute(text("SELECT id FROM notification_rule_targets WHERE rule_id = :rid"), {"rid": rule_id}).fetchone()
            if target_exists:
                conn.execute(text("""
                    UPDATE notification_rule_targets
                    SET camera_id = :cam, zone_id = :zone, class_id = :cls
                    WHERE id = :tid
                """), {"cam": cam_id, "zone": zone_id, "cls": class_id, "tid": target_exists[0]})
            else:
                conn.execute(text("""
                    INSERT INTO notification_rule_targets (rule_id, camera_id, zone_id, class_id)
                    VALUES (:rid, :cam, :zone, :cls)
                """), {"rid": rule_id, "cam": cam_id, "zone": zone_id, "cls": class_id})
                
            if recipient_group_name:
                grp_res = conn.execute(text("SELECT id FROM recipient_groups WHERE name = :name"), {"name": recipient_group_name}).fetchone()
                if grp_res:
                    grp_id = grp_res[0]
                    link_exists = conn.execute(text("SELECT id FROM notification_rule_recipients WHERE rule_id = :rid AND group_id = :gid"),
                                               {"rid": rule_id, "gid": grp_id}).fetchone()
                    if not link_exists:
                        conn.execute(text("INSERT INTO notification_rule_recipients (rule_id, group_id) VALUES (:rid, :gid)"),
                                     {"rid": rule_id, "gid": grp_id})
            elif recipient_emails:
                grp_name = f"{rule_name} Recipient Group"
                grp_res = conn.execute(text("SELECT id FROM recipient_groups WHERE name = :name"), {"name": grp_name}).fetchone()
                if grp_res:
                    grp_id = grp_res[0]
                else:
                    grp_id = conn.execute(text("INSERT INTO recipient_groups (name, description) VALUES (:name, '') RETURNING id"), {"name": grp_name}).scalar()
                
                for email in recipient_emails:
                    rec_exists = conn.execute(text("SELECT id FROM recipients WHERE group_id = :gid AND recipient = :rec AND channel = 'email'"),
                                              {"gid": grp_id, "rec": email}).fetchone()
                    if not rec_exists:
                        conn.execute(text("INSERT INTO recipients (group_id, channel, recipient) VALUES (:gid, 'email', :rec)"),
                                     {"gid": grp_id, "rec": email})
                                     
                link_exists = conn.execute(text("SELECT id FROM notification_rule_recipients WHERE rule_id = :rid AND group_id = :gid"),
                                           {"rid": rule_id, "gid": grp_id}).fetchone()
                if not link_exists:
                    conn.execute(text("INSERT INTO notification_rule_recipients (rule_id, group_id) VALUES (:rid, :gid)"),
                                 {"rid": rule_id, "gid": grp_id})
                                 
            return {"success": True, "message": f"Successfully configured notification rule '{rule_name}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to configure notification rule: {str(e)}"}
@tool
def delete_rule_by_name(rule_type: Literal["detection", "hse", "notification"], rule_name_or_id: str, camera_name_or_id: Optional[str] = None) -> Dict[str, Any]:
    """
    Deletes a specified rule (detection task, HSE safety rule, or notification alert trigger).
    Required fields: rule_type, rule_name_or_id.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not rule_type:
        return {"success": False, "error": "Missing required field: rule_type. Please specify rule_type as 'detection', 'hse', or 'notification'."}
    if not rule_name_or_id:
        return {"success": False, "error": "Missing required field: rule_name_or_id. Please ask the operator for the rule name or ID."}
        
    try:
        with engine.begin() as conn:
            if rule_type == "detection":
                cam_id = None
                if camera_name_or_id:
                    cam_id = _resolve_camera_id(conn, camera_name_or_id)
                mod_id = _resolve_model_id(conn, rule_name_or_id)
                if mod_id:
                    sql = "DELETE FROM detection_assignments WHERE model_id = :mod"
                    params = {"mod": mod_id}
                    if cam_id:
                        sql += " AND camera_id = :cam"
                        params["cam"] = cam_id
                    conn.execute(text(sql), params)
                    return {"success": True, "message": f"Successfully deleted detection assignment rule."}
                else:
                    return {"success": False, "error": f"AI model '{rule_name_or_id}' not found."}
            elif rule_type == "hse":
                rule_id = _resolve_rule_id(conn, rule_name_or_id)
                if not rule_id:
                    return {"success": False, "error": f"HSE safety rule definition '{rule_name_or_id}' not found."}
                sql = "DELETE FROM hse_camera_rules WHERE rule_id = :rid"
                params = {"rid": rule_id}
                if camera_name_or_id:
                    cam_id = _resolve_camera_id(conn, camera_name_or_id)
                    if cam_id:
                        sql += " AND camera_id = :cam"
                        params["cam"] = cam_id
                conn.execute(text(sql), params)
                return {"success": True, "message": f"Successfully deleted HSE safety rule '{rule_name_or_id}'."}
            elif rule_type == "notification":
                res = conn.execute(text("SELECT id FROM notification_rules WHERE name = :name"), {"name": rule_name_or_id}).fetchone()
                if not res:
                    return {"success": False, "error": f"Notification rule '{rule_name_or_id}' not found."}
                rid = res[0]
                conn.execute(text("DELETE FROM notification_rule_recipients WHERE rule_id = :rid"), {"rid": rid})
                conn.execute(text("DELETE FROM notification_rule_targets WHERE rule_id = :rid"), {"rid": rid})
                conn.execute(text("DELETE FROM notification_rules WHERE id = :rid"), {"rid": rid})
                return {"success": True, "message": f"Successfully deleted notification rule '{rule_name_or_id}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to delete rule: {str(e)}"}
@tool
def update_notification_settings(
    smtp_host: Optional[str] = None,
    smtp_port: Optional[int] = None,
    smtp_username: Optional[str] = None,
    smtp_password: Optional[str] = None,
    smtp_from_email: Optional[str] = None,
    smtp_use_tls: Optional[bool] = None,
    telegram_bot_token: Optional[str] = None,
    teams_webhook_url: Optional[str] = None,
    whatsapp_account_sid: Optional[str] = None,
    whatsapp_auth_token: Optional[str] = None,
    whatsapp_from_number: Optional[str] = None
) -> Dict[str, Any]:
    """
    Updates the system SMTP server credentials, Telegram bot tokens, or webhook connections.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    
    try:
        with engine.begin() as conn:
            res = conn.execute(text("SELECT id FROM notification_settings LIMIT 1")).fetchone()
            
            updates = {}
            if smtp_host is not None: updates["smtp_host"] = smtp_host
            if smtp_port is not None: updates["smtp_port"] = smtp_port
            if smtp_username is not None: updates["smtp_username"] = smtp_username
            if smtp_password is not None: updates["smtp_password"] = smtp_password
            if smtp_from_email is not None: updates["smtp_from_email"] = smtp_from_email
            if smtp_use_tls is not None: updates["smtp_use_tls"] = smtp_use_tls
            if telegram_bot_token is not None: updates["telegram_bot_token"] = telegram_bot_token
            if teams_webhook_url is not None: updates["teams_webhook_url"] = teams_webhook_url
            if whatsapp_account_sid is not None: updates["whatsapp_account_sid"] = whatsapp_account_sid
            if whatsapp_auth_token is not None: updates["whatsapp_auth_token"] = whatsapp_auth_token
            if whatsapp_from_number is not None: updates["whatsapp_from_number"] = whatsapp_from_number
            
            if not updates:
                return {"success": False, "error": "No update parameters provided. Please specify SMTP, Telegram, or Webhook properties."}
                
            if res:
                set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                updates["id"] = res[0]
                conn.execute(text(f"UPDATE notification_settings SET {set_clause} WHERE id = :id"), updates)
            else:
                columns = ", ".join(updates.keys())
                placeholders = ", ".join([f":{k}" for k in updates.keys()])
                conn.execute(text(f"INSERT INTO notification_settings ({columns}) VALUES ({placeholders})"), updates)
                
            return {"success": True, "message": "Successfully updated system notification settings."}
    except Exception as e:
        return {"success": False, "error": f"Failed to update settings: {str(e)}"}
@tool
def manage_scheduled_report(
    action: Literal["create", "update", "delete"],
    report_id: Optional[int] = None,
    frequency: Optional[Literal["daily", "weekly", "monthly"]] = None,
    send_time: Optional[str] = None,
    day_of_week: Optional[int] = None,
    day_of_month: Optional[int] = None,
    format: Optional[Literal["pdf", "excel", "csv"]] = None,
    email_recipients: Optional[List[str]] = None,
    camera_name_or_id: Optional[str] = None,
    zone_name: Optional[str] = None
) -> Dict[str, Any]:
    """
    Creates, updates, or deletes scheduled reporting configurations.
    Required fields for creation: action, frequency, send_time, format.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    
    try:
        with engine.begin() as conn:
            if action == "delete":
                if not report_id:
                    res = conn.execute(text("SELECT id FROM scheduled_reports LIMIT 2")).fetchall()
                    if len(res) == 1:
                        report_id = res[0][0]
                    else:
                        return {"success": False, "error": "Missing required field: report_id. Please ask the operator for the report ID to delete."}
                conn.execute(text("DELETE FROM scheduled_reports WHERE id = :id"), {"id": report_id})
                return {"success": True, "message": f"Successfully deleted scheduled report ID {report_id}."}
                
            cam_id = None
            if camera_name_or_id:
                cam_id = _resolve_camera_id(conn, camera_name_or_id)
            zone_id = None
            if zone_name and cam_id:
                zone_id = _resolve_zone_id(conn, cam_id, zone_name)
                
            if action == "create":
                if not frequency:
                    return {"success": False, "error": "Missing required field: frequency. Please specify if report frequency is daily, weekly, or monthly."}
                if not send_time:
                    return {"success": False, "error": "Missing required field: send_time (e.g. 18:00). Please specify the report send time."}
                if not format:
                    return {"success": False, "error": "Missing required field: format (pdf, excel, csv). Please specify the report format."}
                    
                conn.execute(text("""
                    INSERT INTO scheduled_reports (channels, email_recipients, frequency, day_of_week, day_of_month, send_time, format, camera_id, zone_id, is_active)
                    VALUES (:chans, :recips, :freq, :dow, :dom, :time, :fmt, :cam, :zone, true)
                """), {
                    "chans": json.dumps(["email"]),
                    "recips": json.dumps(email_recipients or []),
                    "freq": frequency,
                    "dow": day_of_week,
                    "dom": day_of_month,
                    "time": send_time,
                    "fmt": format,
                    "cam": cam_id,
                    "zone": zone_id
                })
                return {"success": True, "message": "Successfully created scheduled report."}
            elif action == "update":
                if not report_id:
                    res = conn.execute(text("SELECT id FROM scheduled_reports LIMIT 2")).fetchall()
                    if len(res) == 1:
                        report_id = res[0][0]
                    else:
                        return {"success": False, "error": "Missing required field: report_id. Please specify which report ID to update."}
                        
                updates = {}
                if frequency is not None: updates["frequency"] = frequency
                if send_time is not None: updates["send_time"] = send_time
                if day_of_week is not None: updates["day_of_week"] = day_of_week
                if day_of_month is not None: updates["day_of_month"] = day_of_month
                if format is not None: updates["format"] = format
                if email_recipients is not None: updates["email_recipients"] = json.dumps(email_recipients)
                if cam_id is not None: updates["camera_id"] = cam_id
                if zone_id is not None: updates["zone_id"] = zone_id
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["id"] = report_id
                    conn.execute(text(f"UPDATE scheduled_reports SET {set_clause} WHERE id = :id"), updates)
                return {"success": True, "message": f"Successfully updated scheduled report ID {report_id}."}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage scheduled report: {str(e)}"}
@tool
def manage_user(
    action: Literal["create", "update", "deactivate", "reset_password"],
    username: str,
    full_name: Optional[str] = None,
    email: Optional[str] = None,
    role_name: Optional[str] = None,
    password: Optional[str] = None,
    department_name: Optional[str] = None,
    designation_name: Optional[str] = None
) -> Dict[str, Any]:
    """
    Creates, updates, deactivates, or resets credentials for user operator profiles.
    Required fields for creation: action, username, full_name, email, role_name.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not username:
        return {"success": False, "error": "Missing required field: username. Please ask the operator for the username."}
        
    try:
        with engine.begin() as conn:
            if action == "deactivate":
                conn.execute(text("UPDATE users SET is_active = false WHERE username = :uname"), {"uname": username})
                return {"success": True, "message": f"Successfully deactivated user '{username}'."}
                
            role_id = None
            if role_name:
                r_res = conn.execute(text("SELECT id FROM roles WHERE name ILIKE :name"), {"name": role_name}).fetchone()
                if r_res:
                    role_id = r_res[0]
                else:
                    return {"success": False, "error": f"Role '{role_name}' not found."}
                    
            dept_id = None
            if department_name:
                d_res = conn.execute(text("SELECT id FROM departments WHERE name ILIKE :name"), {"name": department_name}).fetchone()
                if d_res:
                    dept_id = d_res[0]
                else:
                    dept_id = conn.execute(text("INSERT INTO departments (name, description) VALUES (:name, '') RETURNING id"), {"name": department_name}).scalar()
                    
            desig_id = None
            if designation_name:
                des_res = conn.execute(text("SELECT id FROM designations WHERE name ILIKE :name"), {"name": designation_name}).fetchone()
                if des_res:
                    desig_id = des_res[0]
                else:
                    desig_id = conn.execute(text("INSERT INTO designations (name, department_id) VALUES (:name, :did) RETURNING id"),
                                           {"name": designation_name, "did": dept_id}).scalar()
                                           
            if action == "create":
                if not full_name:
                    return {"success": False, "error": "Missing required field: full_name. Please ask the operator for the user's full name."}
                if not email:
                    return {"success": False, "error": "Missing required field: email. Please ask the operator for the user's email."}
                if not role_id:
                    return {"success": False, "error": "Missing required field: role_name (e.g. Administrator, Safety Operator). Please ask the operator for the user's role."}
                    
                pwd_hash = password or "pbkdf2:sha256:default_hashed_password"
                conn.execute(text("""
                    INSERT INTO users (full_name, email, username, password_hash, role_id, is_active, department_id, designation_id)
                    VALUES (:name, :email, :uname, :pwd, :role, true, :dept, :desig)
                """), {
                    "name": full_name,
                    "email": email,
                    "uname": username,
                    "pwd": pwd_hash,
                    "role": role_id,
                    "dept": dept_id,
                    "desig": desig_id
                })
                return {"success": True, "message": f"Successfully created user '{username}'."}
            elif action == "reset_password":
                if not password:
                    return {"success": False, "error": "Missing required field: password. Please ask the operator for the new password."}
                conn.execute(text("UPDATE users SET password_hash = :pwd WHERE username = :uname"), {"pwd": password, "uname": username})
                return {"success": True, "message": f"Successfully reset password for user '{username}'."}
            elif action == "update":
                updates = {}
                if full_name is not None: updates["full_name"] = full_name
                if email is not None: updates["email"] = email
                if role_id is not None: updates["role_id"] = role_id
                if dept_id is not None: updates["department_id"] = dept_id
                if desig_id is not None: updates["designation_id"] = desig_id
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["uname"] = username
                    conn.execute(text(f"UPDATE users SET {set_clause} WHERE username = :uname"), updates)
                return {"success": True, "message": f"Successfully updated user '{username}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage user profile: {str(e)}"}
@tool
def manage_camera(
    action: Literal["create", "update", "enable", "disable", "delete"],
    camera_name_or_id: str,
    new_name: Optional[str] = None,
    ip: Optional[str] = None,
    port: Optional[int] = None,
    camera_number: Optional[str] = None,
    rtsp_template: Optional[str] = None,
    stream_type: Optional[str] = None,
    password: Optional[str] = None
) -> Dict[str, Any]:
    """
    Creates, updates, enables, or disables camera hardware setups.
    Required fields for creation: camera_name_or_id, ip, port.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not camera_name_or_id:
        return {"success": False, "error": "Missing required field: camera_name_or_id. Please specify camera name or ID."}
        
    try:
        with engine.begin() as conn:
            cam_id = _resolve_camera_id(conn, camera_name_or_id)
            
            if action == "delete":
                if not cam_id:
                    return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
                # Delete associated records to avoid foreign key violations
                conn.execute(text("DELETE FROM camera_status_logs WHERE camera_id = :id"), {"id": cam_id})
                conn.execute(text("DELETE FROM detection_assignments WHERE camera_id = :id"), {"id": cam_id})
                conn.execute(text("DELETE FROM hse_camera_rules WHERE camera_id = :id"), {"id": cam_id})
                conn.execute(text("DELETE FROM zones WHERE camera_id = :id"), {"id": cam_id})
                conn.execute(text("DELETE FROM counting_configs WHERE camera_id = :id"), {"id": cam_id})
                conn.execute(text("DELETE FROM scheduled_reports WHERE camera_id = :id"), {"id": cam_id})
                conn.execute(text("DELETE FROM cameras WHERE id = :id"), {"id": cam_id})
                return {"success": True, "message": f"Successfully deleted camera '{camera_name_or_id}'."}
                
            if action == "disable":
                if not cam_id:
                    return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
                conn.execute(text("UPDATE cameras SET status = 'offline' WHERE id = :id"), {"id": cam_id})
                return {"success": True, "message": f"Successfully disabled camera '{camera_name_or_id}'."}
            elif action == "enable":
                if not cam_id:
                    return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
                conn.execute(text("UPDATE cameras SET status = 'online' WHERE id = :id"), {"id": cam_id})
                return {"success": True, "message": f"Successfully enabled camera '{camera_name_or_id}'."}
                
            if action == "create":
                if not ip:
                    return {"success": False, "error": "Missing required field: ip. Please ask the operator for the IP address."}
                if port is None:
                    return {"success": False, "error": "Missing required field: port. Please ask the operator for the port number."}
                
                # Fetch user_id from jupyter notebook globals
                notebook_user_id = str(globals().get("user_id", "admin"))
                    
                conn.execute(text("""
                    INSERT INTO cameras (name, ip, port, camera_number, rtsp_template, stream_type, status, user_id, password)
                    VALUES (:name, :ip, :port, :num, :rtsp, :stream, 'online', :user_id, :password)
                """), {
                    "name": camera_name_or_id,
                    "ip": ip,
                    "port": port,
                    "num": camera_number or camera_name_or_id,
                    "rtsp": rtsp_template or "{camera_number}",
                    "stream": stream_type or "main",
                    "user_id": notebook_user_id,
                    "password": password
                })
                
                # Fetch the newly created record to return
                res_cam = conn.execute(text("""
                    SELECT id, name, ip, port, camera_number, rtsp_template, stream_type, status, user_id, password
                    FROM cameras
                    WHERE name = :name
                    ORDER BY id DESC LIMIT 1
                """), {"name": camera_name_or_id}).fetchone()
                
                record_data = dict(res_cam._mapping) if res_cam else {}
                return {
                    "success": True,
                    "message": f"Successfully created camera '{camera_name_or_id}'.",
                    "record": record_data
                }
            elif action == "update":
                if not cam_id:
                    return {"success": False, "error": f"Camera '{camera_name_or_id}' not found."}
                updates = {}
                if new_name is not None: updates["name"] = new_name
                if ip is not None: updates["ip"] = ip
                if port is not None: updates["port"] = port
                if camera_number is not None: updates["camera_number"] = camera_number
                if rtsp_template is not None: updates["rtsp_template"] = rtsp_template
                if stream_type is not None: updates["stream_type"] = stream_type
                if password is not None: updates["password"] = password
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["id"] = cam_id
                    conn.execute(text(f"UPDATE cameras SET {set_clause} WHERE id = :id"), updates)
                
                # Fetch the updated record to return
                res_cam = conn.execute(text("""
                    SELECT id, name, ip, port, camera_number, rtsp_template, stream_type, status, user_id, password
                    FROM cameras
                    WHERE id = :id
                """), {"id": cam_id}).fetchone()
                
                record_data = dict(res_cam._mapping) if res_cam else {}
                return {
                    "success": True,
                    "message": f"Successfully updated camera '{camera_name_or_id}'.",
                    "record": record_data
                }
    except Exception as e:
        return {"success": False, "error": f"Failed to manage camera: {str(e)}"}
@tool
def manage_ai_model(
    action: Literal["create", "update", "delete", "activate", "deactivate"],
    model_name_or_id: str,
    version: Optional[str] = None,
    framework: Optional[str] = None,
    model_path: Optional[str] = None,
    description: Optional[str] = None
) -> Dict[str, Any]:
    """
    Creates, updates, deletes, activates, or deactivates AI models in the vision system.
    Required fields for creation: model_name_or_id, model_path.
    """
    if engine is None: return {"success": False, "error": "Database connector is uninitialized."}
    if not model_name_or_id:
        return {"success": False, "error": "Missing required field: model_name_or_id. Please specify the AI model name or ID."}
        
    try:
        with engine.begin() as conn:
            mod_id = _resolve_model_id(conn, model_name_or_id)
            
            if action == "delete":
                if not mod_id:
                    return {"success": False, "error": f"AI model '{model_name_or_id}' not found."}
                conn.execute(text("DELETE FROM ai_model_classes WHERE model_id = :id"), {"id": mod_id})
                conn.execute(text("DELETE FROM detection_assignments WHERE model_id = :id"), {"id": mod_id})
                conn.execute(text("DELETE FROM ai_models WHERE id = :id"), {"id": mod_id})
                return {"success": True, "message": f"Successfully deleted AI model '{model_name_or_id}'."}
            elif action == "activate":
                if not mod_id:
                    return {"success": False, "error": f"AI model '{model_name_or_id}' not found."}
                conn.execute(text("UPDATE ai_models SET is_active = true WHERE id = :id"), {"id": mod_id})
                return {"success": True, "message": f"Successfully activated AI model '{model_name_or_id}'."}
            elif action == "deactivate":
                if not mod_id:
                    return {"success": False, "error": f"AI model '{model_name_or_id}' not found."}
                conn.execute(text("UPDATE ai_models SET is_active = false WHERE id = :id"), {"id": mod_id})
                return {"success": True, "message": f"Successfully deactivated AI model '{model_name_or_id}'."}
                
            if action == "create":
                if not model_path:
                    return {"success": False, "error": "Missing required field: model_path. Please ask the operator for the file storage path of the model."}
                conn.execute(text("""
                    INSERT INTO ai_models (name, version, framework, model_path, description, is_active)
                    VALUES (:name, :version, :framework, :path, :desc, true)
                """), {
                    "name": model_name_or_id,
                    "version": version or "v1.0",
                    "framework": framework or "PyTorch",
                    "path": model_path,
                    "desc": description or ""
                })
                return {"success": True, "message": f"Successfully created AI model '{model_name_or_id}'."}
            elif action == "update":
                if not mod_id:
                    return {"success": False, "error": f"AI model '{model_name_or_id}' not found."}
                updates = {}
                if version is not None: updates["version"] = version
                if framework is not None: updates["framework"] = framework
                if model_path is not None: updates["model_path"] = model_path
                if description is not None: updates["description"] = description
                
                if updates:
                    set_clause = ", ".join([f"{k} = :{k}" for k in updates.keys()])
                    updates["id"] = mod_id
                    conn.execute(text(f"UPDATE ai_models SET {set_clause} WHERE id = :id"), updates)
                return {"success": True, "message": f"Successfully updated AI model '{model_name_or_id}'."}
    except Exception as e:
        return {"success": False, "error": f"Failed to manage AI model: {str(e)}"}
@tool
def acknowledge_telemetry_flags(
    target_type: Literal["alerts", "incidents", "anomaly_flags", "agent_recommendations"],
    record_ids: List[int],
    action_type: Literal["acknowledge", "dismiss"] = "acknowledge"
) -> Dict[str, Any]:
    """
    TELEMETRY HANDLER: Quick macro to acknowledge or dismiss active system warnings,
    safety anomaly flags, or computer vision detection incidents.
    """
    if engine is None: 
        return {"success": False, "error": "Database connector is uninitialized."}
    if not record_ids:
        return {"success": False, "error": "No valid target record IDs provided."}
        
    column_flag = "is_dismissed" if action_type == "dismiss" and target_type == "agent_recommendations" else "is_acknowledged"
    sql = f"UPDATE {target_type} SET {column_flag} = true WHERE id IN :id_list;"
    
    try:
        with engine.begin() as conn:
            conn.execute(text(sql), {"id_list": tuple(record_ids)})
            return {
                "success": True,
                "message": f"Successfully marked IDs {record_ids} as {action_type}d inside [{target_type}]."
            }
    except Exception as e:
        return {"success": False, "error": f"Failed to execute telemetry change: {str(e)}"}
# Global assignment array explicitly tracking Setup Agent components
setup_agent_tools_registry = [
    manage_recipient_group,
    manage_recipient,
    create_or_update_zone,
    delete_zone_by_name,
    create_or_update_detection_assignment,
    create_or_update_hse_rule,
    create_or_update_notification_rule,
    delete_rule_by_name,
    update_notification_settings,
    manage_scheduled_report,
    manage_user,
    manage_camera,
    manage_ai_model,
    acknowledge_telemetry_flags
]


# Video Agent


In [ ]:
# ====================================================================
# 🛠️ VIDEO AGENT TOOLS  (Production-Ready + Live-Stream Data Gathering)
# ====================================================================
from typing import Dict, Any, List, Optional
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from sqlalchemy import text
from pinecone import Pinecone, ServerlessSpec
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import json
import os
import time
import base64
import logging
import cv2
import numpy as np
from datetime import datetime

logger = logging.getLogger("video_agent_tools")

# Global in-memory cache for Stateful Visual Question Answering (VQA)
SCENE_CACHE: Dict[Any, Dict[str, Any]] = {}

# Cache the loaded YOLO model across calls instead of reloading it on every tool invocation
_YOLO_MODEL_CACHE: Dict[str, Any] = {}

# Where live snapshots get persisted for later evidence/audit trail
SNAPSHOT_DIR = os.getenv("VIDEO_SNAPSHOT_DIR", "storage/live_snapshots")
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

# RTSP frame-grab timeout (ms) so a dead camera never hangs the whole agent mesh
RTSP_OPEN_TIMEOUT_MS = 8000
RTSP_READ_TIMEOUT_MS = 8000

# ====================================================================
# 🔌 PINECONE (LONG-TERM SEMANTIC MEMORY OF SCENES)
# ====================================================================
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc_client = None
pinecone_index = None

if pinecone_api_key:
    try:
        pc_client = Pinecone(api_key=pinecone_api_key)
        index_name = "industrial-safety-summaries"

        active_indexes = [idx.name for idx in pc_client.list_indexes()]
        if index_name not in active_indexes:
            print(f"Creating Pinecone index '{index_name}'...")
            pc_client.create_index(
                name=index_name,
                dimension=768,  # models/text-embedding-004 is 768 dimensions
                metric="cosine",
                spec=ServerlessSpec(cloud="aws", region="us-east-1"),
            )
            while not pc_client.describe_index(index_name).status.ready:
                time.sleep(1)

        pinecone_index = pc_client.Index(index_name)
        print(f"Pinecone connected to index '{index_name}' successfully!")
    except Exception as e:
        print(f"Failed to initialize Pinecone: {e}")


# ====================================================================
# 🧰 SHARED INTERNAL HELPERS (DB + CAMERA + MODEL + CAPTURE)
# ====================================================================
def _lookup_camera_record(camera_name_or_id: Optional[str] = None) -> Any:
    """Fetches every camera row, or resolves a single one by id/number/name. DB-first, always."""
    if engine is None:
        return {"error": "Database connector is uninitialized."}
    with engine.connect() as conn:
        rows = conn.execute(text(
            "SELECT id, name, ip, port, camera_number, rtsp_template, password, status, user_id "
            "FROM cameras ORDER BY id;"
        )).fetchall()
        all_cams = [dict(r._mapping) for r in rows]

    if camera_name_or_id is None:
        return all_cams

    query_val = str(camera_name_or_id).strip().lower()
    for c in all_cams:
        if query_val in {str(c["id"]).lower(), str(c["camera_number"]).lower()} or query_val in str(c["name"]).lower():
            return c

    return {
        "error": f"Camera '{camera_name_or_id}' not found.",
        "available_cameras": [f"'{c['name']}' (ID: {c['id']}, Number: {c['camera_number']})" for c in all_cams],
    }


def _build_rtsp_url(camera_data: Dict[str, Any]) -> str:
    template = camera_data.get("rtsp_template") or str(camera_data.get("camera_number"))
    user = camera_data.get("user_id") or "admin"
    password = camera_data.get("password") or "admin"
    return f"rtsp://{user}:{password}@{camera_data['ip']}:{camera_data['port']}/{template}"


def _capture_frame(camera_data: Dict[str, Any]):
    """Opens the RTSP stream with bounded timeouts and grabs a single fresh frame."""
    rtsp_url = _build_rtsp_url(camera_data)
    cap = cv2.VideoCapture(rtsp_url, cv2.CAP_FFMPEG)
    cap.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, RTSP_OPEN_TIMEOUT_MS)
    cap.set(cv2.CAP_PROP_READ_TIMEOUT_MSEC, RTSP_READ_TIMEOUT_MS)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

    if not cap.isOpened():
        cap.release()
        return None, f"Could not connect to RTSP stream for camera '{camera_data['name']}' at {camera_data['ip']}:{camera_data['port']}."

    ret, frame = cap.read()
    cap.release()
    if not ret or frame is None:
        return None, f"Connected to camera '{camera_data['name']}' but failed to read a live frame."
    return frame, None


def _get_yolo_model(model_name_filter: str = "Base Model 2"):
    """Loads (and caches) the YOLO model path configured in the database for the requested model name."""
    if model_name_filter in _YOLO_MODEL_CACHE:
        return _YOLO_MODEL_CACHE[model_name_filter]

    from ultralytics import YOLO

    model_path = "yolov8n.pt"  # safe default fallback
    try:
        if engine is not None:
            with engine.connect() as conn:
                model_row = conn.execute(
                    text("SELECT model_path FROM ai_models WHERE name ILIKE :name AND is_active = true LIMIT 1"),
                    {"name": model_name_filter},
                ).fetchone()
                if model_row and model_row[0]:
                    db_path_norm = model_row[0].replace("\\", "/")
                    if "storage/models" in db_path_norm:
                        rel = db_path_norm[db_path_norm.find("storage/models"):]
                        if os.path.exists(rel):
                            model_path = rel
                    elif os.path.exists(db_path_norm):
                        model_path = db_path_norm
    except Exception as e:
        logger.warning(f"Falling back to default YOLO weights, DB model lookup failed: {e}")

    model = YOLO(model_path)
    _YOLO_MODEL_CACHE[model_name_filter] = model
    return model


def _frame_to_base64_jpeg(frame) -> str:
    _, buffer = cv2.imencode(".jpg", frame)
    return base64.b64encode(buffer).decode("utf-8")


def _run_yolo_counts(frame) -> Dict[str, int]:
    model = _get_yolo_model()
    results = model(frame, conf=0.25, verbose=False)
    counts: Dict[str, int] = {}
    for result in results:
        for box in result.boxes:
            cls_name = model.names[int(box.cls[0])]
            counts[cls_name] = counts.get(cls_name, 0) + 1
    return counts


def _embed_text(query: str):
    embeddings_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004", google_api_key=os.getenv("GEMINI_API_KEY"))
    return embeddings_model.embed_query(query)


def _summary_to_plain_text(summary: Dict[str, Any]) -> str:
    """
    Defensively flattens whatever shape the VLM scene JSON came back in.
    The VLM is not guaranteed to always emit the exact same keys, so this
    NEVER assumes a fixed schema (this was the root cause of the earlier
    'Failed to index summary to Pinecone: people' crash).
    """
    if not isinstance(summary, dict):
        return str(summary)[:1500]

    parts = []
    for key in ("scene_summary", "scene_type", "people_count", "activities", "machines",
                "objects", "hazards", "environment", "ppe_compliance", "important_observations"):
        if key in summary and summary[key] not in (None, "", [], {}):
            value = summary[key]
            if isinstance(value, (list, dict)):
                value = json.dumps(value, ensure_ascii=False)
            parts.append(f"{key}: {value}")

    if not parts:
        # Totally unknown schema - fall back to a safe, truncated raw dump
        parts.append(json.dumps(summary, ensure_ascii=False)[:1200])

    return "\n".join(parts)


def _index_summary_to_pinecone(camera_id: int, camera_name: str, summary: Dict[str, Any]) -> None:
    if not pinecone_index:
        return
    try:
        summary_text = f"Camera {camera_name} (ID: {camera_id}) Scene Context Summary.\n{_summary_to_plain_text(summary)}"
        vector = _embed_text(summary_text)
        upsert_id = f"cam_{camera_id}_{int(datetime.now().timestamp())}"
        metadata = {
            "camera_id": camera_id,
            "camera_name": camera_name,
            "created_at": str(datetime.now()),
            "summary_text": summary_text,
        }
        pinecone_index.upsert(vectors=[(upsert_id, vector, metadata)])
    except Exception as pe:
        logger.warning(f"Failed to index summary to Pinecone: {pe}")


def _log_camera_status(camera_id: int, status: str, extra: Optional[str] = None) -> None:
    """Writes a row into camera_status_logs so live-health checks are fully DB-integrated / auditable."""
    if engine is None:
        return
    try:
        with engine.begin() as conn:
            conn.execute(text(
                "INSERT INTO camera_status_logs (camera_id, status, checked_at) VALUES (:cid, :status, NOW())"
            ), {"cid": camera_id, "status": status})
    except Exception as e:
        logger.warning(f"Could not write camera_status_logs entry: {e}")


# ====================================================================
# 🛠️ TOOL 1 — LIVE STREAM URL RESOLUTION
# ====================================================================
@tool
def get_video_stream_url(camera_name_or_id: str, duration_seconds: int = 10) -> Dict[str, Any]:
    """
    Retrieves the streaming URL / feed endpoint for a given camera, resolved fully from the
    database (ip, port, camera_number, credentials), for live viewing or downstream analysis.
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"success": False, **(camera_data if isinstance(camera_data, dict) else {"error": "Ambiguous camera reference."})}

    stream_url = _build_rtsp_url(camera_data)
    return {
        "success": True,
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "status": camera_data["status"],
        "stream_url": stream_url,
        "duration_requested_seconds": duration_seconds,
        "message": f"Successfully generated streaming endpoint URL for camera '{camera_data['name']}'.",
    }


# ====================================================================
# 🛠️ TOOL 2 — SINGLE-FRAME OBJECT COUNTING (YOLO)
# ====================================================================
@tool
def analyze_video_feed(camera_name_or_id: str, target_class: str = "person", confidence_threshold: float = 0.25) -> Dict[str, Any]:
    """
    Connects to the specified camera's live RTSP stream, captures a fresh frame, and uses the
    active YOLO detection model (resolved from the database) to count occurrences of target_class.
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"success": False, **(camera_data if isinstance(camera_data, dict) else {"error": "Ambiguous camera reference."})}

    frame, err = _capture_frame(camera_data)
    if err:
        return {"success": False, "error": err}

    try:
        all_detections = _run_yolo_counts(frame)
    except Exception as e:
        return {"success": False, "error": f"Error running inference: {str(e)}"}

    target_lower = target_class.lower().strip()
    detections_count = sum(v for k, v in all_detections.items() if k.lower() == target_lower)

    return {
        "success": True,
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "camera_number": camera_data["camera_number"],
        "target_class": target_class,
        "count": detections_count,
        "all_detections": all_detections,
        "message": f"Found {detections_count} {target_class}(s) on camera '{camera_data['name']}'.",
    }


# ====================================================================
# 🛠️ TOOL 3 — STATEFUL VISUAL QUESTION ANSWERING (VQA), CACHED
# ====================================================================
@tool
def analyze_scene_context(camera_name_or_id: str, operator_question: str) -> Dict[str, Any]:
    """
    STATEFUL VISUAL QUESTION ANSWERING (VQA):
    Answers natural language questions about the live CCTV camera scene while optimizing vision
    LLM costs. Uses an intelligent scene cache, refreshed on TTL expiry (300s), an explicit
    refresh phrase, or a cheap YOLO-count change-detection check.
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"error": camera_data.get("error") if isinstance(camera_data, dict) else "Ambiguous camera reference."}

    camera_id = camera_data["id"]

    refresh_keywords = ["refresh", "live view", "current status", "latest scene", "live", "now", "current"]
    is_force_refresh = any(kw in operator_question.lower() for kw in refresh_keywords)

    ttl_seconds = 300
    cached_entry = SCENE_CACHE.get(camera_id)
    cache_hit = False
    if cached_entry and not is_force_refresh:
        elapsed = (datetime.now() - cached_entry["created_at"]).total_seconds()
        cache_hit = elapsed < ttl_seconds

    def _answer_from_cache(entry, source_label):
        prompt = f"""
        You are a safety assistant. Answer the operator's question based strictly on the cached
        semantic summary of the scene.

        CACHED SEMANTIC SUMMARY:
        {json.dumps(entry['summary'], indent=2, ensure_ascii=False)}

        OPERATOR QUESTION: {operator_question}
        """
        text_response = base_llm.invoke([SystemMessage(content=prompt)])
        return {
            "source": source_label,
            "camera_id": camera_data["id"],
            "camera_name": camera_data["name"],
            "created_at": str(entry["created_at"]),
            "scene_version": entry["scene_version"],
            "summary": entry["summary"],
            "answer": text_response.content,
        }

    if cache_hit:
        try:
            return _answer_from_cache(cached_entry, "cache")
        except Exception as e:
            return {"error": f"Failed to answer from cache: {str(e)}"}

    # Cache miss / forced refresh -> capture a live frame
    frame, err = _capture_frame(camera_data)
    if err:
        return {"error": err}

    try:
        current_detections = _run_yolo_counts(frame)
    except Exception as e:
        current_detections = {}
        logger.warning(f"YOLO change-detection pass failed, continuing with full VLM refresh: {e}")

    if cached_entry and current_detections and current_detections == cached_entry.get("last_frame_detections"):
        cached_entry["created_at"] = datetime.now()
        try:
            return _answer_from_cache(cached_entry, "cache_reuse_no_change")
        except Exception as e:
            return {"error": f"Failed to answer from reused cache: {str(e)}"}

    vlm_prompt = """
You are the Vision Intelligence Engine for an Industrial AI Surveillance Platform. Build a
factual, reusable semantic memory of this scene (never hallucinate, never invent objects).
Return ONLY valid JSON (no markdown fences needed, but they are tolerated) with this shape:

{
  "scene_type": "",
  "scene_summary": "150-250 word factual description covering people, activities, machines, hazards, PPE, environment",
  "people_count": 0,
  "people": [{"id": "person_1", "location": "", "activity": "", "clothing": "", "ppe": []}],
  "activities": [],
  "machines": [{"name": "", "status": "Operating|Idle|Unknown"}],
  "objects": [],
  "hazards": [],
  "environment": {"lighting": "", "cleanliness": ""},
  "ppe_compliance": {"helmet_missing": [], "vest_missing": []},
  "important_observations": []
}

Only report visible facts. If uncertain, write "Unknown".
"""
    try:
        base64_image = _frame_to_base64_jpeg(frame)
        image_message = HumanMessage(content=[
            {"type": "text", "text": vlm_prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
        ])
        vlm_response = gemini_llm.invoke([image_message])

        clean_content = vlm_response.content.strip()
        if clean_content.startswith("```json"):
            clean_content = clean_content[7:]
        if clean_content.startswith("```"):
            clean_content = clean_content[3:]
        if clean_content.endswith("```"):
            clean_content = clean_content[:-3]
        clean_content = clean_content.strip()

        new_summary = json.loads(clean_content)
    except Exception as e:
        return {"error": f"Failed live VQA vision analysis: {str(e)}"}

    version = (cached_entry["scene_version"] + 1) if cached_entry else 1
    SCENE_CACHE[camera_id] = {
        "camera_id": str(camera_data["id"]),
        "camera_name": camera_data["name"],
        "created_at": datetime.now(),
        "scene_version": version,
        "summary": new_summary,
        "last_frame_detections": current_detections,
    }

    # Long-term semantic memory (schema-agnostic, safe against missing keys)
    _index_summary_to_pinecone(camera_id, camera_data["name"], new_summary)

    try:
        prompt = f"""
        You are a safety assistant. Answer the operator's question based strictly on the semantic
        summary of the scene.

        SEMANTIC SUMMARY:
        {json.dumps(new_summary, indent=2, ensure_ascii=False)}

        OPERATOR QUESTION: {operator_question}
        """
        text_response = base_llm.invoke([SystemMessage(content=prompt)])
    except Exception as e:
        return {"error": f"Vision analysis succeeded but answer generation failed: {str(e)}"}

    return {
        "source": "live_vision_refresh",
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "created_at": str(SCENE_CACHE[camera_id]["created_at"]),
        "scene_version": SCENE_CACHE[camera_id]["scene_version"],
        "summary": SCENE_CACHE[camera_id]["summary"],
        "answer": text_response.content,
    }


# ====================================================================
# 🛠️ TOOL 4 — SEMANTIC SEARCH OVER HISTORICAL SCENE MEMORY (PINECONE)
# ====================================================================
@tool
def semantic_search_scene_history(query: str, camera_name_or_id: Optional[str] = None) -> Dict[str, Any]:
    """
    Performs a semantic vector similarity search across past CCTV scene summaries stored in
    Pinecone. Use for historical patterns/anomalies (e.g. "show all helmet violations in Room 6").
    """
    if not pinecone_index:
        return {"success": False, "error": "Pinecone vector index is uninitialized or unavailable."}

    try:
        target_camera_name = None
        if camera_name_or_id:
            camera_data = _lookup_camera_record(camera_name_or_id)
            if isinstance(camera_data, dict) and "name" in camera_data:
                target_camera_name = camera_data["name"]

        query_vector = _embed_text(query)
        filter_dict = {"camera_name": target_camera_name} if target_camera_name else None

        results = pinecone_index.query(vector=query_vector, top_k=5, include_metadata=True, filter=filter_dict)

        matches = [{
            "score": match.get("score"),
            "camera_name": match.get("metadata", {}).get("camera_name"),
            "created_at": match.get("metadata", {}).get("created_at"),
            "summary": match.get("metadata", {}).get("summary_text"),
        } for match in results.get("matches", [])]

        return {
            "success": True,
            "query": query,
            "filter_camera": target_camera_name,
            "matches": matches,
            "message": f"Successfully retrieved {len(matches)} matching historical scene summaries from Pinecone.",
        }
    except Exception as e:
        return {"success": False, "error": f"Failed semantic history search: {str(e)}"}


# ====================================================================
# 🛠️ TOOL 5 — LIVE STREAM HEALTH CHECK  (NEW, DB-integrated)
# ====================================================================
@tool
def get_live_stream_health(camera_name_or_id: Optional[str] = None) -> List[Dict[str, Any]]:
    """
    Probes live RTSP connectivity for one camera, or every camera if none is specified, measures
    connect latency, and logs the result into camera_status_logs for historical uptime tracking.
    """
    cameras = _lookup_camera_record(camera_name_or_id)
    if isinstance(cameras, dict) and "error" in cameras:
        return [cameras]
    if isinstance(cameras, dict):
        cameras = [cameras]

    report = []
    for cam in cameras:
        start = time.perf_counter()
        frame, err = _capture_frame(cam)
        latency_ms = round((time.perf_counter() - start) * 1000, 1)
        status = "online" if frame is not None else "offline"
        _log_camera_status(cam["id"], status, err)
        report.append({
            "camera_id": cam["id"],
            "camera_name": cam["name"],
            "live_status": status,
            "latency_ms": latency_ms,
            "error": err,
        })
    return report


# ====================================================================
# 🛠️ TOOL 6 — LIVE SNAPSHOT CAPTURE (NEW, evidence/audit trail)
# ====================================================================
@tool
def capture_live_snapshot(camera_name_or_id: str, reason: Optional[str] = None) -> Dict[str, Any]:
    """
    Captures a single live frame from the camera's RTSP stream right now, saves it to disk as a
    timestamped JPEG evidence snapshot, and returns the file path plus a base64 thumbnail.
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"success": False, **(camera_data if isinstance(camera_data, dict) else {"error": "Ambiguous camera reference."})}

    frame, err = _capture_frame(camera_data)
    if err:
        return {"success": False, "error": err}

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"cam{camera_data['id']}_{timestamp}.jpg"
    filepath = os.path.join(SNAPSHOT_DIR, filename)
    cv2.imwrite(filepath, frame)

    return {
        "success": True,
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "snapshot_path": filepath,
        "captured_at": datetime.now().isoformat(),
        "reason": reason or "manual_operator_request",
        "thumbnail_base64": _frame_to_base64_jpeg(cv2.resize(frame, (320, 180))),
        "message": f"Snapshot saved for camera '{camera_data['name']}' at {filepath}.",
    }


# ====================================================================
# 🛠️ TOOL 7 — MULTI-CAMERA LIVE PEOPLE COUNT (NEW, fleet-wide)
# ====================================================================
@tool
def get_live_people_count_multi_camera(camera_names_or_ids: Optional[List[str]] = None, target_class: str = "person") -> Dict[str, Any]:
    """
    Runs live YOLO detection across multiple cameras at once (or the entire fleet if no list is
    given) and returns a per-camera and total live headcount for target_class right now.
    """
    if camera_names_or_ids:
        cameras = []
        for ref in camera_names_or_ids:
            cam = _lookup_camera_record(ref)
            if isinstance(cam, dict) and "id" in cam:
                cameras.append(cam)
    else:
        cameras = _lookup_camera_record(None)
        if isinstance(cameras, dict):
            cameras = []

    per_camera = []
    total = 0
    for cam in cameras:
        frame, err = _capture_frame(cam)
        if err:
            per_camera.append({"camera_id": cam["id"], "camera_name": cam["name"], "count": None, "error": err})
            continue
        try:
            counts = _run_yolo_counts(frame)
        except Exception as e:
            per_camera.append({"camera_id": cam["id"], "camera_name": cam["name"], "count": None, "error": str(e)})
            continue
        count = sum(v for k, v in counts.items() if k.lower() == target_class.lower())
        total += count
        per_camera.append({"camera_id": cam["id"], "camera_name": cam["name"], "count": count})

    return {
        "success": True,
        "target_class": target_class,
        "total_live_count": total,
        "per_camera": per_camera,
        "cameras_checked": len(per_camera),
        "checked_at": datetime.now().isoformat(),
    }


# ====================================================================
# 🛠️ TOOL 8 — LIVE MOTION DETECTION (NEW)
# ====================================================================
@tool
def detect_motion_in_stream(camera_name_or_id: str, sample_interval_seconds: float = 1.0, sensitivity: float = 25.0) -> Dict[str, Any]:
    """
    Grabs two live frames a short interval apart from the same camera and computes a frame
    difference to determine whether motion is currently happening in the scene.
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"success": False, **(camera_data if isinstance(camera_data, dict) else {"error": "Ambiguous camera reference."})}

    frame1, err1 = _capture_frame(camera_data)
    if err1:
        return {"success": False, "error": err1}
    time.sleep(max(0.2, min(sample_interval_seconds, 5.0)))
    frame2, err2 = _capture_frame(camera_data)
    if err2:
        return {"success": False, "error": err2}

    try:
        gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
        gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
        gray1 = cv2.GaussianBlur(gray1, (21, 21), 0)
        gray2 = cv2.GaussianBlur(gray2, (21, 21), 0)
        delta = cv2.absdiff(gray1, gray2)
        thresh = cv2.threshold(delta, sensitivity, 255, cv2.THRESH_BINARY)[1]
        motion_pixel_ratio = float(np.count_nonzero(thresh)) / thresh.size
    except Exception as e:
        return {"success": False, "error": f"Motion analysis failed: {str(e)}"}

    return {
        "success": True,
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "motion_detected": motion_pixel_ratio > 0.02,
        "motion_score_percent": round(motion_pixel_ratio * 100, 2),
        "sample_interval_seconds": sample_interval_seconds,
        "checked_at": datetime.now().isoformat(),
    }


# ====================================================================
# 🛠️ TOOL 9 — FIND PERSON BY DESCRIPTION, LIVE (NEW)
# ====================================================================
@tool
def find_person_by_description_live(camera_name_or_id: str, description: str) -> Dict[str, Any]:
    """
    Captures a fresh live frame and asks the vision model whether anyone matching a free-text
    description (e.g. "person in a pink shirt", "someone without a helmet") is currently visible,
    and if so, where in the frame they are located.
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"success": False, **(camera_data if isinstance(camera_data, dict) else {"error": "Ambiguous camera reference."})}

    frame, err = _capture_frame(camera_data)
    if err:
        return {"success": False, "error": err}

    prompt = f"""
You are a live CCTV search assistant. Look at this frame ONLY and answer factually.
Question: Is there a person matching this description currently visible: "{description}"?
Respond ONLY as JSON: {{"found": true/false, "location": "e.g. foreground center / near desk 2 / Unknown", "confidence": "high|medium|low", "note": "one short factual sentence"}}
Never hallucinate. If uncertain, set found to false and confidence to "low".
"""
    try:
        base64_image = _frame_to_base64_jpeg(frame)
        image_message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
        ])
        vlm_response = gemini_llm.invoke([image_message])
        clean = vlm_response.content.strip().strip("`")
        if clean.lower().startswith("json"):
            clean = clean[4:].strip()
        result = json.loads(clean)
    except Exception as e:
        return {"success": False, "error": f"Live description search failed: {str(e)}"}

    return {
        "success": True,
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "description_query": description,
        "checked_at": datetime.now().isoformat(),
        **result,
    }


# ====================================================================
# 🛠️ TOOL 10 — LIVE PPE COMPLIANCE CHECK  (NEW, writes to anomaly_flags)
# ====================================================================
@tool
def get_live_ppe_compliance_check(camera_name_or_id: str, zone_name: Optional[str] = None) -> Dict[str, Any]:
    """
    Runs a live PPE (helmet/vest) compliance check against the camera's current frame. If a
    violation is detected, it is written into the anomaly_flags table so it shows up in the
    System Agent's alert and reporting tools (full database integration, not just chat output).
    """
    camera_data = _lookup_camera_record(camera_name_or_id)
    if isinstance(camera_data, list) or "error" in camera_data:
        return {"success": False, **(camera_data if isinstance(camera_data, dict) else {"error": "Ambiguous camera reference."})}

    frame, err = _capture_frame(camera_data)
    if err:
        return {"success": False, "error": err}

    prompt = """
You are a PPE compliance inspector for an industrial safety camera. Examine this frame and report
ONLY visible facts as JSON:
{"people_count": 0, "helmet_missing_count": 0, "vest_missing_count": 0, "compliant": true, "notes": ""}
If uncertain about any person, do not count them as missing PPE. Never hallucinate.
"""
    try:
        base64_image = _frame_to_base64_jpeg(frame)
        image_message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
        ])
        vlm_response = gemini_llm.invoke([image_message])
        clean = vlm_response.content.strip().strip("`")
        if clean.lower().startswith("json"):
            clean = clean[4:].strip()
        result = json.loads(clean)
    except Exception as e:
        return {"success": False, "error": f"PPE compliance analysis failed: {str(e)}"}

    violation = not result.get("compliant", True) and (result.get("helmet_missing_count", 0) or result.get("vest_missing_count", 0))

    if violation and engine is not None:
        try:
            zone_id = None
            if zone_name:
                with engine.connect() as conn:
                    zone_id = _resolve_zone_id(conn, camera_data["id"], zone_name)
            with engine.begin() as conn:
                conn.execute(text("""
                    INSERT INTO anomaly_flags
                        (anomaly_type, severity, description, zone_id, camera_id, class_name,
                         observed_value, event_count, is_acknowledged, created_at)
                    VALUES
                        ('ppe_violation', 'high', :desc, :zone_id, :camera_id, 'ppe_missing',
                         :observed, 1, false, NOW())
                """), {
                    "desc": result.get("notes") or "Live PPE compliance check flagged missing PPE.",
                    "zone_id": zone_id,
                    "camera_id": camera_data["id"],
                    "observed": json.dumps(result),
                })
        except Exception as e:
            logger.warning(f"Live PPE violation detected but failed to log to anomaly_flags: {e}")

    return {
        "success": True,
        "camera_id": camera_data["id"],
        "camera_name": camera_data["name"],
        "checked_at": datetime.now().isoformat(),
        "violation_logged_to_db": bool(violation and engine is not None),
        **result,
    }


# ====================================================================
# 📋 VIDEO AGENT TOOL REGISTRY
# ====================================================================
video_agent_tools_registry = [
    get_video_stream_url,
    analyze_video_feed,
    analyze_scene_context,
    semantic_search_scene_history,
    get_live_stream_health,
    capture_live_snapshot,
    get_live_people_count_multi_camera,
    detect_motion_in_stream,
    find_person_by_description_live,
    get_live_ppe_compliance_check,
]


In [ ]:
reasoning_groq_llm = ChatLiteLLM(
    model="groq/llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.0,
    max_tokens=2000,
)

reasoning_gemini_llm = ChatLiteLLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.0,
    max_tokens=2000,
)
# Ensure your analytical reasoning framework matches the swap
reasoning_llm = FallbackLLM([
    ("Gemini", reasoning_gemini_llm),
    ("Groq", reasoning_groq_llm)
])



# Tools provide to Agents

In [ ]:
# ====================================================================
# INITIALIZE AGENT LLM BINDINGS 
# ====================================================================

# 1. System Agent - Needs maximum reasoning precision to perform dynamic read-only DB querying
system_llm = reasoning_llm.bind_tools(system_agent_tools_registry)

# 2. Setup Agent - Standard generation temperature to craft flexible INSERT / UPDATE payload structures
setup_llm = base_llm.bind_tools(setup_agent_tools_registry)

# 3. General Agent - Completely sandboxed from database write permissions for absolute safety
general_llm = base_llm.bind_tools(general_agent_tools_registry)

# 4. Video Agent - Binds video tools
video_llm = reasoning_llm.bind_tools(video_agent_tools_registry)

In [ ]:
# ====================================================================
# SYSTEM CAPABILITIES REGISTRY 
# ====================================================================
system_capabilities = """
1. General Agent: Greets you, answers system feature questions, and explains capabilities.
2. System Agent: Reads telemetry data, camera health, safety alerts, counting logs, and reports.
3. Setup Agent: Creates, modifies, and deletes cameras, zones, notification rules, and schedules.
4. Investigator Agent: Analyzes timeline data to provide root-cause autopsies and summaries for safety incidents.
5. Video Agent: Retrieves streaming feed URLs and video footage endpoints for cameras to view live video feeds.
6. Supervisor Router: Analyzes queries to delegate tasks to the correct specialized agent.
7. Deletion Guardrail: The system asks you for confirmation before executing any delete actions.
"""


In [ ]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage, ToolMessage
from typing import Literal
from langgraph.graph import END
import re
from datetime import datetime
def _fetch_user_profile_info(user_id_str: str) -> str:
    if engine is None:
        return "User Profile Info: Database uninitialized."
    try:
        with engine.connect() as conn:
            query = text("""
                SELECT 
                    u.full_name, 
                    u.email, 
                    u.username, 
                    r.name AS role_name,
                    d.name AS department_name, 
                    des.name AS designation_title
                FROM users u
                LEFT JOIN roles r ON r.id = u.role_id
                LEFT JOIN departments d ON d.id = u.department_id
                LEFT JOIN designations des ON des.id = u.designation_id
                WHERE u.id = :id OR u.username = :uname OR u.email = :email;
            """)
            params = {
                "id": None,
                "uname": user_id_str,
                "email": user_id_str
            }
            try:
                params["id"] = int(user_id_str)
            except ValueError:
                pass
                
            row = conn.execute(query, params).fetchone()
            if row:
                full_name, email, username, role_name, dept_name, desig_title = row
                return (
                    f"User Profile Info:\n"
                    f"- Full Name: {full_name or 'N/A'}\n"
                    f"- Username: {username or 'N/A'}\n"
                    f"- Email: {email or 'N/A'}\n"
                    f"- Role: {role_name or 'N/A'}\n"
                    f"- Department: {dept_name or 'N/A'}\n"
                    f"- Designation: {desig_title or 'N/A'}"
                )
            return "User Profile Info: Not found in database."
    except Exception as e:
        return f"User Profile Info: Failed to retrieve ({e})."
# ====================================================================
# 🚀 CORE PLATFORM AGENT NODES (FIXED SINGLE-TURN TERMINATION)
# ====================================================================
def system_agent_node(state: TeamState):
    """
    System Agent Node (Read-Only)
    Extracts telemetry parameters, aggregates station throughput volumes,
    and runs diagnostic logs queries on your camera fleet.
    """
    allowed_comps = get_user_permissions(state.get("user_id", "admin"))
    
    # Extract original user query
    last_human_query = ""
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query:
        req_comp = get_required_component_for_query(last_human_query)
        if req_comp and req_comp not in allowed_comps:
            denied_msg = (
                "Sorry, you don't have permission to access this feature. "
                "Your current role does not have access to this module. "
                "Please contact your administrator if you require this permission."
            )
            return {
                "messages": [AIMessage(content=denied_msg)],
                "next_agent": "FINISH"
            }
            
    # Extract & Preserve Date Context
    date_match = None
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", msg.content)
            if m:
                date_match = m.group(1)
                break
    if not date_match and state.get("current_investigated_date"):
        date_match = state["current_investigated_date"]
    elif not date_match:
        date_match = datetime.now().strftime("%Y-%m-%d")
        
    current_date_context = ""
    if date_match:
        current_date_context = f"\nACTIVE DISCUSSION DATE CONTEXT: {date_match}. If the user references 'this date', 'that day', or asks a follow-up question (e.g. 'in camera 6'), query the database for alerts/incidents strictly on this active date context."
    user_id = 1
    user_info = _fetch_user_profile_info(user_id)
    system_prompt = SystemMessage(content=f"""
    You are the Industrial Vision System Agent. Your job is to fetch telemetry metrics, look up camera status histories,
    summarize production batch counting configurations, and read active zone parameters from the platform.
    
    {current_date_context}
    
    You have absolute read-only permissions. Do not attempt to update or create configurations.
    Always format data summaries into simple, clean markdown tables for the operator.
    
    CRITICAL NO-INCIDENT FORMATTING RULE:
    If no safety incidents, alerts, or rule violations are found matching the query context (e.g. empty tables or empty results), you MUST respond with a normal conversational text response. Do NOT output any JSON blocks, markdown code blocks containing JSON, or structure your answer with null/empty fields.
    
    CRITICAL DELETION CONFIRMATION RULE:
    Before calling any tool with a delete action, you MUST check the conversation history and confirm deletion.
    
    CURRENT AUTHENTICATED USER CONTEXT:
    {user_info}
    """)
    
    allowed_tools = [t for t in system_agent_tools_registry if TOOL_TO_COMPONENT_MAP.get(t.name) in allowed_comps]
    if allowed_tools:
        node_llm = reasoning_llm.bind_tools(allowed_tools)
    else:
        node_llm = reasoning_llm
        
    if state["messages"]:
        last_msg = state["messages"][-1]
        input_content = getattr(last_msg, "content", "")
        if isinstance(input_content, str):
            input_lower = input_content.lower()
            system_blocked_inputs = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
            for pattern in system_blocked_inputs:
                if pattern in input_lower:
                    return {
                        "messages": [AIMessage(content="🚨 [SECURITY ENFORCEMENT]: Request blocked. Reading core authentication strings via chat interface is barred.")],
                        "next_agent": "FINISH"
                    }
                    
    response = node_llm.invoke([system_prompt] + state["messages"])
    
    if hasattr(response, "content") and response.content:
        content_lower = response.content.lower()
        unauthorized_keywords = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token", "teams_webhook_url"]
        for keyword in unauthorized_keywords:
            if keyword in content_lower:
                response.content = "⚠️ [SECURITY ENFORCEMENT]: Response blocked. The agent attempted to output internal system secrets or credentials."
                break
                
    if hasattr(response, "tool_calls") and response.tool_calls:
        # Route back to ourselves to execute tools and resume reasoning
        return {"messages": [response], "next_agent": "system_agent", "current_investigated_date": date_match}
        
    return {"messages": [response], "next_agent": "FINISH", "current_investigated_date": date_match}
def setup_agent_node(state: TeamState):
    """
    Setup Agent Node (Write/Update Mutations)
    Modifies active zone shapes, updates notification recipient profiles,
    and acknowledges active computer vision anomaly alert signals.
    """
    user_id = state.get("user_id", "admin")
    allowed_comps = get_user_permissions(state.get("user_id", "admin"))
    
    last_human_query = ""
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query:
        req_comp = get_required_component_for_query(last_human_query)
        if req_comp and req_comp not in allowed_comps:
            denied_msg = (
                "Sorry, you don't have permission to access this feature. "
                "Your current role does not have access to this module. "
                "Please contact your administrator if you require this permission."
            )
            return {
                "messages": [AIMessage(content=denied_msg)],
                "next_agent": "FINISH"
            }
            
    # Extract & Preserve Date Context
    date_match = None
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", msg.content)
            if m:
                date_match = m.group(1)
                break
    if not date_match and state.get("current_investigated_date"):
        date_match = state["current_investigated_date"]
    elif not date_match:
        date_match = datetime.now().strftime("%Y-%m-%d")
        
    current_date_context = ""
    if date_match:
        current_date_context = f"\nACTIVE DISCUSSION DATE CONTEXT: {date_match}."
    user_info = _fetch_user_profile_info(user_id)
    system_prompt = SystemMessage(content=f"""
    You are the Industrial Vision Setup Agent. Your job is to alter tracking parameters, modify configuration maps,
    insert notification rule targets, update schedules, and acknowledge or dismiss system alert flags.
    
    {current_date_context}
    
    You focus on mutation work. Confirm row IDs are valid integers before processing requests.
    
    DATABASE CAMERA SCHEMA RULES:
    When the operator asks to create or configure a new camera, use manage_camera. Only ask for parameters that actually belong to the camera schema (name, ip, port, camera_number, password, rtsp_template, stream_type).
    
    JSON CONFIRMATION RULE:
    Whenever you successfully create or update any record via tool execution, display details in JSON.
    
    CRITICAL RULE FOR DATA MUTATIONS:
    Before calling any tool, confirm the operator has provided all details.
    
    CRITICAL DELETION CONFIRMATION RULE:
    Confirm deletion before executing a delete.
    
    CURRENT AUTHENTICATED USER CONTEXT:
    {user_info}
    """)
    
    allowed_tools = [t for t in setup_agent_tools_registry if TOOL_TO_COMPONENT_MAP.get(t.name) in allowed_comps]
    if allowed_tools:
        node_llm = base_llm.bind_tools(allowed_tools)
    else:
        node_llm = base_llm
        
    if state["messages"]:
        last_msg = state["messages"][-1]
        input_content = getattr(last_msg, "content", "")
        if isinstance(input_content, str):
            input_lower = input_content.lower()
            setup_blocked_inputs = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
            for pattern in setup_blocked_inputs:
                if pattern in input_lower:
                    return {
                        "messages": [AIMessage(content="🚨 [SECURITY ENFORCEMENT]: Request blocked. Changing internal auth hashes or tokens via chat is restricted.")],
                        "next_agent": "FINISH"
                    }
                    
    response = node_llm.invoke([system_prompt] + state["messages"])
    
    if hasattr(response, "content") and response.content:
        content_lower = response.content.lower()
        unauthorized_keywords = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
        for keyword in unauthorized_keywords:
            if keyword in content_lower:
                response.content = "⚠️ [SECURITY ENFORCEMENT]: Operation terminated. Configuration payload attempted to pass sensitive data signatures."
                break
                
    if hasattr(response, "tool_calls") and response.tool_calls:
        return {"messages": [response], "next_agent": "setup_agent", "current_investigated_date": date_match}
        
    return {"messages": [response], "next_agent": "FINISH", "current_investigated_date": date_match}
def general_agent_node(state: TeamState):
    """
    General Agent Node (Sandboxed Conversation)
    Answers operator greetings and questions directly using core memory or user profiles.
    """
    user_id = state.get("user_id", "admin")
    allowed_comps = get_user_permissions(state.get("user_id", "admin"))
    
    last_human_query = ""
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query:
        req_comp = get_required_component_for_query(last_human_query)
        if req_comp and req_comp not in allowed_comps:
            denied_msg = (
                "Sorry, you don't have permission to access this feature. "
                "Your current role does not have access to this module. "
                "Please contact your administrator if you require this permission."
            )
            return {
                "messages": [AIMessage(content=denied_msg)],
                "next_agent": "FINISH"
            }
            
    user_info = _fetch_user_profile_info(user_id)
    system_prompt = SystemMessage(content=f"""
    You are the General Platform Agent. Handle introductions, operator greetings, 
    and answer questions about system features using your core conversational memory.
    
    MULTI-AGENT SYSTEM CAPABILITIES:
    {system_capabilities}
    
    DELEGATION TO SYSTEM AGENT RULE:
    If the user's query requires fetching, searching, or querying database records and you do not have the answer, delegate to the System Agent. Explain you will consult the database, and append the tag [DELEGATION: system_agent] at the very end of your response text.
    
    CRITICAL NO-INCIDENT FORMATTING RULE:
    If no safety incidents, alerts, or rule violations are found matching the query context (e.g., if a sub-agent has reported that no incidents were found), you MUST respond with a normal conversational text response. Do NOT output any JSON blocks, markdown code blocks containing JSON, or structure your answer with null/empty fields.
    
    CURRENT AUTHENTICATED USER CONTEXT:
    {user_info}
    """)
    
    allowed_tools = [t for t in general_agent_tools_registry if TOOL_TO_COMPONENT_MAP.get(t.name) in allowed_comps]
    if allowed_tools:
        node_llm = base_llm.bind_tools(allowed_tools)
    else:
        node_llm = base_llm
        
    response = node_llm.invoke([system_prompt] + state["messages"])
    
    if hasattr(response, "tool_calls") and response.tool_calls:
        return {"messages": [response], "next_agent": "general_agent"}
        
    next_agent = "FINISH"
    if response.content and "[DELEGATION: system_agent]" in response.content:
        next_agent = "system_agent"
        response.content = response.content.replace("[DELEGATION: system_agent]", "").strip()
        
    return {"messages": [response], "next_agent": next_agent}
def investigator_agent_node(state: TeamState):
    """
    Incident Investigator Agent Node (Layer 4 Synthesis Agent)
    Analyzes event timelines and history to determine the root cause of incidents.
    """
    user_id = state.get("user_id", "admin")
    allowed_comps = get_user_permissions(user_id)
    
    last_human_query = ""
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query:
        req_comp = get_required_component_for_query(last_human_query)
        if req_comp and req_comp not in allowed_comps:
            denied_msg = (
                "Sorry, you don't have permission to access this feature. "
                "Your current role does not have access to this module. "
                "Please contact your administrator if you require this permission."
            )
            return {
                "messages": [AIMessage(content=denied_msg)],
                "next_agent": "FINISH"
            }
            
    # Extract & Preserve Date Context
    date_match = None
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", msg.content)
            if m:
                date_match = m.group(1)
                break
    if not date_match and state.get("current_investigated_date"):
        date_match = state["current_investigated_date"]
    elif not date_match:
        date_match = datetime.now().strftime("%Y-%m-%d")
        
    current_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    current_date_prompt = f"CURRENT SYSTEM DATE AND TIME: {current_time_str}. Use this to resolve relative date references."
    
    cached_context = ""
    if date_match and state.get("date_summary_cache"):
        cached_context = (
            f"\nCACHED CONTEXT FOR THE DATE {date_match}:\n"
            f"Here is the daily summary of safety events for this date:\n"
            f"{state['date_summary_cache']}\n"
            f"The user is asking a follow-up query about this date. Use this cached summary instead of calling the database tool again. Only call the database tool if they explicitly query a DIFFERENT date."
        )
        
    user_info = _fetch_user_profile_info(user_id)
    system_prompt = SystemMessage(content=f"""
    You are the Incident Investigator Agent. Your job is to analyze event timelines, histories, and logs to perform root-cause autopsies and forensic summaries on safety incidents.
    
    {current_date_prompt}
    {cached_context}
    
    SYSTEM PROMPT REQUIREMENTS:
    1. To investigate any safety event query, whether broad (search) or deep (autopsy, root cause, timeline reconstruction), you MUST call the `investigate_events` tool passing the operator's request as the `user_query` argument.
    2. If the tool retrieves safety events (incidents, alerts, or violations), you MUST provide a structured JSON response matching the ForensicWizardOutput schema.
    3. If the tool retrieves NO events or incidents (empty results), you MUST respond with a normal natural language conversational text response. Do NOT output any JSON block or structured response containing null or empty fields.
    
    DELEGATION RULE:
    If the user's query requires features or tools of other agents (e.g. system_agent for general database queries/listings, video_agent for live video stream URLs/inference, or general_agent for general chat), you can delegate the task to them.
    To do this, explain who you will consult, and append the tag [DELEGATION: system_agent], [DELEGATION: video_agent], or [DELEGATION: general_agent] at the very end of your response text.
    Do NOT delegate to setup_agent (database write/update operations).
    
    STRICT ANTI-HALLUCINATION GUARDRAIL:
    You MUST ground every single causal statement purely in the retrieved event data, and never invent or assume unseen events.
    """)
    
    allowed_tools = [t for t in investigator_agent_tools_registry if TOOL_TO_COMPONENT_MAP.get(t.name) in allowed_comps]
    if allowed_tools:
        node_llm = reasoning_llm.bind_tools(allowed_tools)
    else:
        node_llm = reasoning_llm
        
    if state["messages"]:
        last_msg = state["messages"][-1]
        input_content = getattr(last_msg, "content", "")
        if isinstance(input_content, str):
            input_lower = input_content.lower()
            setup_blocked_inputs = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
            for pattern in setup_blocked_inputs:
                if pattern in input_lower:
                    return {
                        "messages": [AIMessage(content="🚨 [SECURITY ENFORCEMENT]: Request blocked. Changing internal auth hashes or tokens via chat is restricted.")],
                        "next_agent": "FINISH"
                    }
                    
    response = node_llm.invoke([system_prompt] + state["messages"])
    
    if hasattr(response, "content") and response.content:
        content_lower = response.content.lower()
        unauthorized_keywords = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
        for keyword in unauthorized_keywords:
            if keyword in content_lower:
                response.content = "⚠️ [SECURITY ENFORCEMENT]: Operation terminated. Configuration payload attempted to pass sensitive data signatures."
                break
                
    if response.content and "[DELEGATION:" in response.content:
        next_agent = "FINISH"
        for agent in ["system_agent", "video_agent", "general_agent"]:
            if f"[DELEGATION: {agent}]" in response.content or f"[DELEGATION:{agent}]" in response.content:
                next_agent = agent
                response.content = response.content.replace(f"[DELEGATION: {agent}]", "").replace(f"[DELEGATION:{agent}]", "").strip()
                break
        return {"messages": [response], "next_agent": next_agent, "current_investigated_date": date_match}
    if hasattr(response, "tool_calls") and response.tool_calls:
        return {"messages": [response], "next_agent": "investigator_agent", "current_investigated_date": date_match}
        
    # Check if any database records were actually retrieved
    has_events = False
    for msg in reversed(state["messages"]):
        if isinstance(msg, ToolMessage) or getattr(msg, "type", "") == "tool":
            try:
                data = json.loads(msg.content)
                if isinstance(data, dict):
                    if data.get("investigation_type") == "deep" and data.get("incident"):
                        has_events = True
                    elif data.get("investigation_type") in ["search", "search_fallback"]:
                        if data.get("alerts") or data.get("hse_rule_events") or data.get("incidents"):
                            has_events = True
            except Exception:
                pass
            break
    if not has_events:
        # Return normal text response instead of JSON format when no incidents occur
        return {
            "messages": [response],
            "next_agent": "FINISH",
            "current_investigated_date": date_match
        }
    try:
        structured_llm = base_llm.with_structured_output(ForensicWizardOutput)
        formatting_prompt = SystemMessage(content=(
            "You are a response formatter. Read the conversation history and format the final answer into the structured JSON schema.\n"
            "Constraints:\n"
            "- incident_id: List of matching alert, incident, or HSE event IDs (integers).\n"
            "- summary: A detailed summary of all safety and HSE activity for the complete day, strictly exactly 5 lines.\n"
            "- response: A single-line response addressing the user's specific query based on the matched events.\n"
        ))
        structured_response = structured_llm.invoke([formatting_prompt] + state["messages"] + [response])
        
        final_content = json.dumps(structured_response.dict() if hasattr(structured_response, "dict") else structured_response.model_dump())
        response = AIMessage(content=final_content)
        
        return {
            "messages": [response],
            "next_agent": "FINISH",
            "current_investigated_date": date_match,
            "date_summary_cache": structured_response.summary
        }
    except Exception as e:
        print(f"Error during structured formatting: {e}")
    return {"messages": [response], "next_agent": "FINISH", "current_investigated_date": date_match}
def video_agent_node(state: TeamState):
    """
    Video Agent Node
    Retrieves video stream URLs, handles video feed endpoints, VQA semantic context caching,
    and runs semantic searches on historical cctv scene summaries.
    """
    user_id = state.get("user_id", "admin")
    allowed_comps = get_user_permissions(user_id)
    
    last_human_query = ""
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query:
        req_comp = get_required_component_for_query(last_human_query)
        if req_comp and req_comp not in allowed_comps:
            denied_msg = (
                "Sorry, you don't have permission to access this feature. "
                "Your current role does not have access to this module. "
                "Please contact your administrator if you require this permission."
            )
            return {
                "messages": [AIMessage(content=denied_msg)],
                "next_agent": "FINISH"
            }
            
    # Re-seed global SCENE_CACHE from state to ensure it survives server restarts
    if state.get("current_video_camera") and state.get("video_summary_cache"):
        try:
            import json
            cache_data = state["video_summary_cache"]
            if isinstance(cache_data, str):
                cache_data = json.loads(cache_data)
            
            with engine.connect() as conn:
                cam_id = _resolve_camera_id(conn, state["current_video_camera"])
                if cam_id:
                    SCENE_CACHE[cam_id] = {
                        "camera_id": str(cam_id),
                        "camera_name": state["current_video_camera"],
                        "created_at": datetime.now(),
                        "scene_version": 1,
                        "summary": cache_data,
                        "last_frame_detections": {}
                    }
        except Exception as e:
            print(f"Error restoring SCENE_CACHE: {e}")
    # Extract & Preserve Date Context
    date_match = None
    for msg in reversed(state["messages"]):
        if msg.type == "human" or isinstance(msg, HumanMessage):
            m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", msg.content)
            if m:
                date_match = m.group(1)
                break
    if not date_match and state.get("current_investigated_date"):
        date_match = state["current_investigated_date"]
    elif not date_match:
        date_match = datetime.now().strftime("%Y-%m-%d")
        
    current_date_context = ""
    if date_match:
        current_date_context = f"\nACTIVE DISCUSSION DATE CONTEXT: {date_match}."
    user_info = _fetch_user_profile_info(user_id)
    system_prompt = SystemMessage(content=f"""
    You are the Industrial Vision Video Agent. Your job is to retrieve live video streaming URLs, RTSP video feeds, or playbacks for specified cameras, answer natural-language questions about live video scenes using the VQA scene caching tool, and run semantic similarity searches on historical scene summaries.
    
    Always present stream links or information in a clear and organized format.
    
    SYSTEM PROMPT REQUIREMENTS:
    1. To answer questions about live camera scene content, status, worker activity, or safety details (VQA queries like "What are the workers doing?", "Are they wearing safety vests?", etc.), call the `analyze_scene_context` tool.
    2. To search historical CCTV summaries, PPE violations patterns, or activities from previous times (e.g. "find all helmet violations in Room 6", "show worker activity yesterday"), call the `semantic_search_scene_history` tool.
    
    {current_date_context}
    
    CURRENT AUTHENTICATED USER CONTEXT:
    {user_info}
    """)
    
    allowed_tools = [t for t in video_agent_tools_registry if TOOL_TO_COMPONENT_MAP.get(t.name) in allowed_comps]
    if allowed_tools:
        node_llm = reasoning_llm.bind_tools(allowed_tools)
    else:
        node_llm = reasoning_llm
        
    if state["messages"]:
        last_msg = state["messages"][-1]
        input_content = getattr(last_msg, "content", "")
        if isinstance(input_content, str):
            input_lower = input_content.lower()
            blocked_inputs = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
            for pattern in blocked_inputs:
                if pattern in input_lower:
                    return {
                        "messages": [AIMessage(content="🚨 [SECURITY ENFORCEMENT]: Request blocked. Changing internal auth hashes or tokens via chat is restricted.")],
                        "next_agent": "FINISH"
                    }
                    
    response = node_llm.invoke([system_prompt] + state["messages"])
    
    if hasattr(response, "content") and response.content:
        content_lower = response.content.lower()
        unauthorized_keywords = ["password_hash", "smtp_password", "telegram_bot_token", "whatsapp_auth_token"]
        for keyword in unauthorized_keywords:
            if keyword in content_lower:
                response.content = "⚠️ [SECURITY ENFORCEMENT]: Operation terminated. Configuration payload attempted to pass sensitive data signatures."
                break
                
    if hasattr(response, "tool_calls") and response.tool_calls:
        return {"messages": [response], "next_agent": "video_agent", "current_investigated_date": date_match}
        
    # Check if there is an active VQA tool output to cache in state
    latest_summary = None
    latest_camera = None
    for msg in reversed(state["messages"]):
        if isinstance(msg, ToolMessage) or getattr(msg, "type", "") == "tool":
            if msg.name == "analyze_scene_context":
                try:
                    import json
                    res = json.loads(msg.content)
                    if isinstance(res, dict) and "summary" in res:
                        latest_summary = res["summary"]
                        latest_camera = res.get("camera_name")
                        break
                except Exception as e:
                    print(f"Error parsing ToolMessage: {e}")
    ret = {"messages": [response], "next_agent": "FINISH", "current_investigated_date": date_match}
    if latest_summary and latest_camera:
        ret["video_summary_cache"] = json.dumps(latest_summary) if isinstance(latest_summary, dict) else latest_summary
        ret["current_video_camera"] = latest_camera
        
    return ret
def supervisor_node(state: TeamState):
    """
    Master Router Supervisor
    Dynamically scans conversation history to locate the true human input.
    """
    current_next = state.get("next_agent")
    
    # If the operator has just spoken, ignore stale delegation state from previous turns
    is_new_user_turn = False
    if state.get("messages") and (isinstance(state["messages"][-1], HumanMessage) or getattr(state["messages"][-1], "type", "") == "human"):
        is_new_user_turn = True
    if current_next in ["system_agent", "setup_agent", "general_agent", "video_agent", "investigator_agent"] and not is_new_user_turn:
        print("[Supervisor Delegation Pass-through]: Routing control directly to [" + current_next + "]")
        return {"next_agent": current_next}
        
    messages = state["messages"]
    options = ["system_agent", "setup_agent", "general_agent", "investigator_agent", "video_agent", "FINISH"]
    
    last_human_query = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage) or getattr(msg, "type", "") == "human":
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query and isinstance(last_human_query, str):
        input_lower = last_human_query.lower()
        
        # If we have an active video camera in state, check if the query is a scene follow-up
        if state.get("current_video_camera"):
            scene_followup_keywords = [
                "person", "shirt", "wearing", "doing", "active", "standing", "sitting",
                "operating", "kr rha", "present", "is there", "helmet", "vest", "ppe",
                "color", "colour", "hazard", "unsafe", "what are they", "what is on"
            ]
            if any(k in input_lower for k in scene_followup_keywords):
                return {"next_agent": "video_agent"}
        # Fast-Track Rule D: Investigation detection (High Precedence)
        investigation_indicators = [
            "investigate", "nvestigate", "root cause", "autopsy", "why did it happen", "incident summary",
            "incidents on", "alerts on", "what happened on", "safety summary for", "violations yesterday",
            "helmet violations", "unsafe acts", "red zone intrusions", "safety events today", "safety events yesterday"
        ]
        if any(ind in input_lower for ind in investigation_indicators) or re.search(r"\b\d{4}-\d{2}-\d{2}\b", input_lower):
            return {"next_agent": "investigator_agent"}
        # Fast-Track Rule A: Mutation detection
        setup_indicators = [
            "insert", "update", "acknowledge", "dismiss", "modify rule", "add recipient",
            "create rule", "detect fire", "notify", "rule for", "alert for", "speed violation",
            "abandoned object", "line crossing", "unauthorized person", "send whatsapp",
            "send telegram", "send teams", "send email", "escalation rule", "confidence threshold",
            "alert threshold", "disable", "enable", "delete rule", "rename rule", "assign camera",
            "remove camera", "add camera", "create zone", "update coordinates", "delete zone",
            "rename zone", "create group", "add john", "remove manager", "smtp settings",
            "bot token", "schedule", "delete report", "report format", "add user", "deactivate user",
            "reset password", "change role", "create department", "create designation",
            "camera status", "archive retention", "two-factor", "session timeout", "allow ip",
            "create model", "activate model", "deactivate model", "delete model", "alert cooldown",
            "notification delay", "severity of", "record video", "automatically acknowledge",
            "create", "new", "setup", "configure", "delete", "create camera", "new camera"
        ]
        if any(ind in input_lower for ind in setup_indicators):
            return {"next_agent": "setup_agent"}
        # Fast-Track Rule E: Video agent detection
        video_indicators = [
            "video feed", "stream url", "live feed", "video stream", "play stream", "rtsp feed",
            "camera stream", "watch camera", "count person", "count people", "count persons",
            "how many people", "how many person", "detect person", "detect people", "detect object",
            "analyze video", "analyze stream", "analyze camera", "worker doing", "workers doing",
            "what are they doing", "what is happening on", "anyone in", "who is in", "what is on camera",
            "show feed", "view feed", "view camera", "show camera", "live status", "whats on camera"
        ]
        if any(ind in input_lower for ind in video_indicators):
            return {"next_agent": "video_agent"}
            
        # Fast-Track Rule B: Retrieval detection
        system_indicators = [
            "fetch", "query", "show logs", "view zones", "counting summary", "check health",
            "camera", "zone", "incident", "alert", "anomaly", "risk", "hse", "violation",
            "defect", "counting", "recording", "snapshot", "notification", "report", "setting",
            "user", "recommendation", "timeline", "event", "who entered", "who violated",
            "how many", "list all", "show all", "which are", "is camera", "what is the",
            "what happened", "summarize", "detail"
        ]
        if any(ind in input_lower for ind in system_indicators):
            return {"next_agent": "system_agent"}
            
        # Fast-Track Rule C: Chit-Chat detection
        general_indicators = [
            "hi", "hello", "hey", "good morning", "who am i", "profile", "account",
            "what are you doing", "who are you", "help", "thanks", "thank you", "bro what",
            "what the fuck"
        ]
        if any(ind in input_lower for ind in general_indicators):
            print("[Supervisor Fast-Track]: Operator query identified. Routing directly to General Agent.")
            return {"next_agent": "general_agent"}
            
    system_prompt = SystemMessage(content="""
    You are the Master Router Supervisor. Analyze the user's intent and select a choice:
    - 'system_agent': User wants to read database metrics, look up logs, check camera/zone configuration list, or view charts.
    - 'setup_agent': User wants to update parameters, add configurations, delete/create rules, or edit system settings.
    - 'general_agent': Standard pleasantries, greetings, conversational questions, complaints, frustration, or general non-technical chat.
    - 'video_agent': User wants to retrieve camera live video stream URLs, feed endpoints, see what workers are doing live, or run live object/person detection.
    
    - If the last message from the assistant was asking for confirmation to perform a deletion or modification, and the user replies with confirmation (e.g. "yes", "confirm", "proceed", etc.), route to 'setup_agent' to execute the action.
    - 'FINISH': The task is fully complete.
    
    Reply with ONLY the matching plain-text string option.
    """)
    
    response = base_llm.invoke([system_prompt] + messages)    
    choice = response.content.strip().lower()
    
    next_step = "general_agent"
    for option in options:
        if option.lower() in choice:
            next_step = option
            break
            
    print("[Supervisor Decision]: Route selected -> [" + next_step + "]")
    return {"next_agent": next_step}
def supervisor_router(state: TeamState) -> Literal["system_agent", "setup_agent", "general_agent", "investigator_agent", "video_agent", "__end__"]:
    """
    Master Router - Evaluates supervisor intent and diverts control 
    to the selected specialist or cleanly terminates the graph session.
    """
    target = state.get("next_agent", "FINISH")
    if not isinstance(target, str):
        target = "FINISH"
    target_clean = target.strip().lower()
    if "system" in target_clean:
        return "system_agent"
    elif "setup" in target_clean:
        return "setup_agent"
    elif "general" in target_clean:
        return "general_agent"
    elif "investigat" in target_clean:
        return "investigator_agent"
    elif "video" in target_clean:
        return "video_agent"
    else:
        print("🛑 [Supervisor Router]: Termination state reached. Exiting graph orchestration matrix.")
        return END


In [ ]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage, ToolMessage
from typing import Literal
from langgraph.graph import END
import re
from datetime import datetime
def _fetch_user_profile_info(user_id_str: str) -> str:
    if engine is None:
        return "User Profile Info: Database uninitialized."
    try:
        with engine.connect() as conn:
            user_sql = """
                SELECT u.username, u.email, r.name as role_name 
                FROM users u
                LEFT JOIN roles r ON r.id = u.role_id
                WHERE u.id = :id OR u.username = :uname OR u.email = :email
            """
            params = {"id": None, "uname": str(user_id_str), "email": str(user_id_str)}
            try:
                params["id"] = int(user_id_str)
            except ValueError:
                pass
            row = conn.execute(text(user_sql), params).fetchone()
            if row:
                return f"Authenticated User Profile -> Username: {row[0]}, Email: {row[1]}, Assigned Role: {row[2]}"
            return f"User Profile Info: Profile details not found for ID/Name '{user_id_str}'."
    except Exception as e:
        return f"User Profile Info: Failed to query user context. Error: {str(e)}"
def supervisor_node(state: TeamState):
    """
    Master Router Supervisor
    Dynamically scans conversation history to locate the true human input.
    """
    current_next = state.get("next_agent")
    
    # If the operator has just spoken, ignore stale delegation state from previous turns
    is_new_user_turn = False
    if state.get("messages") and (isinstance(state["messages"][-1], HumanMessage) or getattr(state["messages"][-1], "type", "") == "human"):
        is_new_user_turn = True
    if current_next in ["system_agent", "setup_agent", "general_agent", "video_agent", "investigator_agent"] and not is_new_user_turn:
        print("[Supervisor Delegation Pass-through]: Routing control directly to [" + current_next + "]")
        return {"next_agent": current_next}
        
    messages = state["messages"]
    options = ["system_agent", "setup_agent", "general_agent", "investigator_agent", "video_agent", "FINISH"]
    
    last_human_query = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage) or getattr(msg, "type", "") == "human":
            last_human_query = getattr(msg, "content", "")
            break
            
    if last_human_query and isinstance(last_human_query, str):
        input_lower = last_human_query.lower()
        
        # If we have an active video camera in state, check if the query is a scene follow-up
        if state.get("current_video_camera"):
            scene_followup_keywords = [
                "person", "shirt", "wearing", "doing", "active", "standing", "sitting",
                "operating", "kr rha", "present", "is there", "helmet", "vest", "ppe",
                "color", "colour", "hazard", "unsafe", "what are they", "what is on"
            ]
            if any(k in input_lower for k in scene_followup_keywords):
                return {"next_agent": "video_agent"}
        # Fast-Track Rule D: Investigation detection (High Precedence)
        investigation_indicators = [
            "investigate", "nvestigate", "root cause", "autopsy", "why did it happen", "incident summary",
            "incidents on", "alerts on", "what happened on", "safety summary for", "violations yesterday",
            "helmet violations", "unsafe acts", "red zone intrusions", "safety events today", "safety events yesterday"
        ]
        if any(ind in input_lower for ind in investigation_indicators) or re.search(r"\b\d{4}-\d{2}-\d{2}\b", input_lower):
            return {"next_agent": "investigator_agent"}
        # Fast-Track Rule A: Mutation detection
        setup_indicators = [
            "insert", "update", "acknowledge", "dismiss", "modify rule", "add recipient",
            "create rule", "detect fire", "notify", "rule for", "alert for", "speed violation",
            "abandoned object", "line crossing", "unauthorized person", "send whatsapp",
            "send telegram", "send teams", "send email", "escalation rule", "confidence threshold",
            "alert threshold", "disable", "enable", "delete rule", "rename rule", "assign camera",
            "remove camera", "add camera", "create zone", "update coordinates", "delete zone",
            "rename zone", "create group", "add john", "remove manager", "smtp settings",
            "bot token", "schedule", "delete report", "report format", "add user", "deactivate user",
            "reset password", "change role", "create department", "create designation",
            "camera status", "archive retention", "two-factor", "session timeout", "allow ip",
            "create model", "activate model", "deactivate model", "delete model", "alert cooldown",
            "notification delay", "severity of", "record video", "automatically acknowledge",
            "create", "new", "setup", "configure", "delete", "create camera", "new camera"
        ]
        if any(ind in input_lower for ind in setup_indicators):
            return {"next_agent": "setup_agent"}
        # Fast-Track Rule E: Video agent detection
        video_indicators = [
            "video feed", "stream url", "live feed", "video stream", "play stream", "rtsp feed",
            "camera stream", "watch camera", "count person", "count people", "count persons",
            "how many people", "how many person", "detect person", "detect people", "detect object",
            "analyze video", "analyze stream", "analyze camera", "worker doing", "workers doing",
            "what are they doing", "what is happening on", "anyone in", "who is in", "what is on camera",
            "show feed", "view feed", "view camera", "show camera", "live status", "whats on camera"
        ]
        if any(ind in input_lower for ind in video_indicators):
            return {"next_agent": "video_agent"}
            
        # Fast-Track Rule B: Retrieval detection
        system_indicators = [
            "fetch", "query", "show logs", "view zones", "counting summary", "check health",
            "camera", "zone", "incident", "alert", "anomaly", "risk", "hse", "violation",
            "defect", "counting", "recording", "snapshot", "notification", "report", "setting",
            "user", "recommendation", "timeline", "event", "who entered", "who violated",
            "how many", "list all", "show all", "which are", "is camera", "what is the",
            "what happened", "summarize", "detail"
        ]
        if any(ind in input_lower for ind in system_indicators):
            return {"next_agent": "system_agent"}
            
        # Fast-Track Rule C: Chit-Chat detection
        general_indicators = [
            "hi", "hello", "hey", "good morning", "who am i", "profile", "account",
            "what are you doing", "who are you", "help", "thanks", "thank you", "bro what",
            "what the fuck"
        ]
        if any(ind in input_lower for ind in general_indicators):
            print("[Supervisor Fast-Track]: Operator query identified. Routing directly to General Agent.")
            return {"next_agent": "general_agent"}
            
    system_prompt = SystemMessage(content="""
    You are the Master Router Supervisor. Analyze the user's intent and select a choice:
    - 'system_agent': User wants to read database metrics, look up logs, check camera/zone configuration list, or view charts.
    - 'setup_agent': User wants to update parameters, add configurations, delete/create rules, or edit system settings.
    - 'general_agent': Standard pleasantries, greetings, conversational questions, complaints, frustration, or general non-technical chat.
    - 'video_agent': User wants to retrieve camera live video stream URLs, feed endpoints, see what workers are doing live, or run live object/person detection.
    
    - If the last message from the assistant was asking for confirmation to perform a deletion or modification, and the user replies with confirmation (e.g. "yes", "confirm", "proceed", etc.), route to 'setup_agent' to execute the action.
    - 'FINISH': The task is fully complete.
    
    Reply with ONLY the matching plain-text string option.
    """)
    
    response = base_llm.invoke([system_prompt] + messages)    
    choice = response.content.strip().lower()
    
    next_step = "general_agent"
    for option in options:
        if option.lower() in choice:
            next_step = option
            break
            
    print("[Supervisor Decision]: Route selected -> [" + next_step + "]")
    return {"next_agent": next_step}
def supervisor_router(state: TeamState) -> Literal["system_agent", "setup_agent", "general_agent", "investigator_agent", "video_agent", "__end__"]:
    """
    Master Router - Evaluates supervisor intent and diverts control 
    to the selected specialist or cleanly terminates the graph session.
    """
    target = state.get("next_agent", "FINISH")
    if not isinstance(target, str):
        target = "FINISH"
    target_clean = target.strip().lower()
    if "system" in target_clean:
        return "system_agent"
    elif "setup" in target_clean:
        return "setup_agent"
    elif "general" in target_clean:
        return "general_agent"
    elif "investigat" in target_clean:
        return "investigator_agent"
    elif "video" in target_clean:
        return "video_agent"
    else:
        print("🛑 [Supervisor Router]: Termination state reached. Exiting graph orchestration matrix.")
        return END


In [ ]:
# Master workspace tools union registry used by LangGraph's ToolNode wrapper
all_system_tools = system_agent_tools_registry + general_agent_tools_registry + setup_agent_tools_registry + investigator_agent_tools_registry + video_agent_tools_registry


# Architecture Building

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, ToolMessage
import json

def execute_tools_node(state: TeamState):
    """
    Custom Tool Executor with dynamic RBAC checks.
    Ensures that if a user does not have permission for a tool's component,
    the tool is blocked and a permission denied response is returned.
    """
    user_id = state.get("user_id", "admin")
    allowed_comps = get_user_permissions(user_id)
    
    last_message = state["messages"][-1]
    tool_messages = []
    
    for tool_call in last_message.tool_calls:
        tool_name = tool_call["name"]
        
        # 1. Find the tool
        tool_obj = next((t for t in all_system_tools if t.name == tool_name), None)
        if not tool_obj:
            tool_messages.append(ToolMessage(
                content=f"Error: Tool '{tool_name}' not found.",
                tool_call_id=tool_call["id"]
            ))
            continue
            
        # 2. Check permission
        tool_component = TOOL_TO_COMPONENT_MAP.get(tool_name)
        if tool_component and tool_component not in allowed_comps:
            denied_msg = (
                "Sorry, you don't have permission to access this feature. "
                "Your current role does not have access to this module. "
                "Please contact your administrator if you require this permission."
            )
            tool_messages.append(ToolMessage(
                content=denied_msg,
                tool_call_id=tool_call["id"]
            ))
            continue
            
        # 3. Execute tool
        try:
            result = tool_obj.invoke(tool_call["args"])
            content = json.dumps(result) if not isinstance(result, str) else result
            tool_messages.append(ToolMessage(
                content=content,
                tool_call_id=tool_call["id"]
            ))
        except Exception as e:
            err_msg = str(e)
            import re
            clean_err = re.sub(r'(delete from|drop table|insert into|truncate table)', '[REDACTED_SQL]', err_msg, flags=re.IGNORECASE)
            tool_messages.append(ToolMessage(
                content=f"Error executing tool: {clean_err}",
                tool_call_id=tool_call["id"]
            ))
            
    return {"messages": tool_messages}

workflow = StateGraph(TeamState)

# ====================================================================
# 1. ADD NEW ARCHITECTURE SYSTEM NODES
# ====================================================================
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("system_agent", system_agent_node)
workflow.add_node("setup_agent", setup_agent_node)
workflow.add_node("general_agent", general_agent_node)
workflow.add_node("investigator_agent", investigator_agent_node)
workflow.add_node("video_agent", video_agent_node)
workflow.add_node("execute_tools", execute_tools_node)

# ====================================================================
# 2. UPGRADED ROUTING MECHANICS WITH EDGE-LEVEL GUARDRAILS (FIXED)
# ====================================================================
def post_agent_router(state: TeamState):
    last_message = state["messages"][-1]
    content_lower = getattr(last_message, "content", "")
    content_lower = content_lower.lower() if isinstance(content_lower, str) else ""
    
    # 🛡️ PLATFORM SECURITY FIREWALL BLACKLIST
    unauthorized_keywords = [
        "password_hash", "smtp_password", "telegram_bot_token", 
        "teams_webhook_url", "whatsapp_auth_token", "api_keys", "otp_codes"
    ]
    
    contains_leak = any(keyword in content_lower for keyword in unauthorized_keywords)

    # RULE A: Tool Call Handler
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        if contains_leak:
            print("🚨 [Edge Guardrail]: Tool call blocked due to token extraction signature.")
            last_message.content = "⚠️ [SECURITY ENFORCEMENT]: Request terminated due to an insecure background trigger attempt."
            return "supervisor" 
        return "execute_tools"
        
    # RULE B: Token Leak Fallback Check
    if contains_leak:
        print("🚨 [Edge Guardrail]: Text response blocked due to security token signature.")
        last_message.content = "⚠️ [SECURITY ENFORCEMENT]: Access Denied. Platform guardrails prevent printing raw authorization tokens."
        return "supervisor"
    
    requested_target = state.get("next_agent", "supervisor")
    if requested_target == "FINISH":
        print("🛑 [Edge Router]: Worker text response complete. Halting graph turn and exiting cleanly.")
        return "__end__"
    
    return "supervisor"


def tool_execution_router(state: TeamState):
    target = state.get("next_agent", "supervisor")
    if not isinstance(target, str):
        target = "supervisor"
        
    target_lower = target.lower()
    print(f"🔄 [Tool Loop Fix]: Tool execution complete. Routing telemetry payload data back to -> [{target_lower}]")
    
    if "system" in target_lower: 
        return "system_agent"
    if "setup" in target_lower: 
        return "setup_agent"
    if "general" in target_lower: 
        return "general_agent"
    if "investigat" in target_lower:
        return "investigator_agent"
    if "video" in target_lower:
        return "video_agent"
    
    return "supervisor"

# ====================================================================
# 3. CONNECT STREAMLINED SYMMETRICAL PATHWAYS (GRAPH GRAPH VISUAL FIX)
# ====================================================================
workflow.add_edge(START, "supervisor")

# Main Supervisor Distribution Matrix Map
workflow.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "system_agent": "system_agent",
        "setup_agent": "setup_agent",
        "general_agent": "general_agent",
        "investigator_agent": "investigator_agent",
        "video_agent": "video_agent",
        "__end__": END
    }
)

clean_worker_routes = {
    "execute_tools": "execute_tools", 
    "supervisor": "supervisor",
    "__end__": END
}

workflow.add_conditional_edges("system_agent", post_agent_router, clean_worker_routes)
workflow.add_conditional_edges("setup_agent", post_agent_router, clean_worker_routes)
workflow.add_conditional_edges("general_agent", post_agent_router, clean_worker_routes)
workflow.add_conditional_edges("investigator_agent", post_agent_router, clean_worker_routes)
workflow.add_conditional_edges("video_agent", post_agent_router, clean_worker_routes)

workflow.add_conditional_edges(
    "execute_tools",
    tool_execution_router,
    {
        "system_agent": "system_agent",
        "setup_agent": "setup_agent",
        "general_agent": "general_agent",
        "investigator_agent": "investigator_agent",
        "video_agent": "video_agent",
        "supervisor": "supervisor",
    }
)

# ====================================================================
# 4. COMPILATION
# ====================================================================
Deva_Super_Agent_mesh = workflow.compile(checkpointer=MemorySaver())
print("✅ New Project Multi-Agent Mesh Compiled Successfully with Fixed Edge Mapping Keys!")



In [ ]:
Deva_Super_Agent_mesh

# run_chat

In [ ]:
import time
import sys

def run_chat(agent_instance, thread_id: str = "industrial_perfect_unlocked_101") -> None:
    config = {"configurable": {"thread_id": thread_id}}
    sys.stderr = sys.__stderr__
    
    # Works in terminal, ignored in Jupyter
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(line_buffering=True)
   
    print("=" * 65, flush=True)
    print(f"🚀 Industrial Vision Platform OS Supervisor Mesh Live [HITL Active].", flush=True)
    print(f"🔗 Active Thread Connection ID: {thread_id}", flush=True)
    print("=" * 65, flush=True)
    
    # Prompt for username to establish authenticated context
    username = input("🔑 Enter Username (default: admin): ").strip()
    if not username:
        username = "admin"

    auth_user_id = None
    role_name = "Unknown"
    
    if engine is not None:
        try:
            with engine.connect() as conn:
                user_sql = text("""
                    SELECT id, username, role_id 
                    FROM users 
                    WHERE username = :uname OR email = :email
                """)
                user_row = conn.execute(user_sql, {"uname": username, "email": username}).fetchone()
                if not user_row:
                    print(f"❌ Authentication Failed: User '{username}' does not exist in the database.\n", flush=True)
                    return
                
                uid, uname, role_id = user_row
                
                # Fetch role name
                role_row = conn.execute(text("SELECT name FROM roles WHERE id = :rid"), {"rid": role_id}).fetchone()
                role_name = role_row[0] if role_row else "Unknown"
                
                print(f"✅ Authenticated as user: {uname} (Role: {role_name})\n", flush=True)
                auth_user_id = str(uid)
                globals()["user_id"] = uid # Set notebook global context dynamically
        except Exception as e:
            print(f"❌ Authentication Error: Database lookup failed ({e}).\n", flush=True)
            return
    else:
        print(f"Proceeding with default context for user '{username}'\n", flush=True)
        auth_user_id = username
        role_name = "Default"
        globals()["user_id"] = username # Set notebook global context dynamically

    print("Type 'exit' or 'quit' to securely close your active terminal session.\n", flush=True)

    session_steps = 0
    session_tool_calls = 0
    active_node_history = []
    start_time = None
    first_event_time = None

    while True:
        try:
            graph_state = agent_instance.get_state(config)
            
            if graph_state.next and "execute_tools" in graph_state.next:
                print("\n⚠️  [HUMAN INTERVENTION REQUIRED] ⚠️", flush=True)
                print("An agent is attempting to execute a core database read/write operation loop.", flush=True)
                user_approval = input("👉 Type 'approve' to execute background transaction or 'fix' to modify: ").strip().lower()
               
                if user_approval in ["exit", "quit"]:
                    break
                   
                if user_approval == "approve":
                    print("\n🧠 Resuming workflow with approval signal...", flush=True)
                    inputs = None
                else:
                    user_feedback = input("📝 Provide explicit fix instructions for the agent team: ").strip()
                    active_node = graph_state.next[0] if graph_state.next else "supervisor"
                    
                    agent_instance.update_state(
                        config,
                        {"messages": [("user", f"Operator rejected execution. Changes required: {user_feedback}")]},
                        as_node=active_node
                    )
                    print("\n🔄 Injection complete. Re-routing task pipeline...", flush=True)
                    inputs = None
            else:
                user_input = input("👤 Operator: ").strip()
                if not user_input:
                    continue
                if user_input.lower() in ["exit", "quit"]:
                    break

                inputs = {
                    "messages": [("user", user_input)],
                    "user_id": auth_user_id
                }
                print("\n🧠 Supervisor orchestrating pipeline...", flush=True)
                
                session_steps = 0
                session_tool_calls = 0
                active_node_history = []
                start_time = time.perf_counter()
                first_event_time = None

            stream_generator = agent_instance.stream(inputs, config, stream_mode="updates")
            
            for chunk in stream_generator:
                session_steps += 1
                if first_event_time is None:
                    first_event_time = time.perf_counter()

                for node_name, node_data in chunk.items():
                    active_node_history.append(node_name)
                    if isinstance(node_data, dict) and "messages" in node_data:
                        for msg in node_data["messages"]:
                            if hasattr(msg, "tool_calls") and msg.tool_calls:
                                session_tool_calls += len(msg.tool_calls)

            post_stream_state = agent_instance.get_state(config)
            
            if post_stream_state.next and "execute_tools" in post_stream_state.next:
                continue

            end_time = time.perf_counter()
            total_time = (end_time - start_time) if start_time else 0.0
            ttft = (first_event_time - start_time) if (first_event_time and start_time) else total_time

            latest_messages = post_stream_state.values.get("messages", [])

            if latest_messages:
                text_messages = [m for m in latest_messages if hasattr(m, 'content') and m.content and m.type != "tool"]
                last_msg = text_messages[-1] if text_messages else latest_messages[-1]
                
                human_messages = [m for m in latest_messages if m.type == "human"]
                display_user_query = human_messages[-1].content if human_messages else "Unknown Input Trigger"
                
                final_agent = "supervisor"
                if active_node_history:
                    filtered_history = [n for n in active_node_history if n not in ["execute_tools", "supervisor"]]
                    if filtered_history:
                        final_agent = filtered_history[-1]

                display_content = getattr(last_msg, "content", "")
                if not isinstance(display_content, str):
                    display_content = str(display_content)

                print("=" * 65, flush=True)                
                print(f"✨ Operator Input: {display_user_query}", flush=True)
                print("=" * 65, flush=True)                
                print(f"\n🤖 [{final_agent.upper()} AGENT RESPONSE]:", flush=True)
                print(display_content.strip(), flush=True)
                print("=" * 65, flush=True)
                
                session_costs = base_llm.gateway_metrics.get("accumulated_cost_usd", 0.0)
                failed_routes = base_llm.gateway_metrics.get("failed_calls", 0)

                input_tokens = output_tokens = total_tokens = "-"
                if hasattr(last_msg, "usage_metadata") and last_msg.usage_metadata:
                    usage = last_msg.usage_metadata
                    input_tokens = usage.get("input_tokens", "-")
                    output_tokens = usage.get("output_tokens", "-")
                    total_tokens = usage.get("total_tokens", "-")

                print(f"⏱️  Metrics      | Time: {total_time:.2f}s | TTFT: {ttft:.2f}s | Steps: {session_steps} | Tool Calls: {session_tool_calls}", flush=True)
                print(f"💰 Gateway Spend | Tokens -> In: {input_tokens} Out: {output_tokens} Total: {total_tokens} | Total Session Spend: ${session_costs:.6f} | API Failovers: {failed_routes}", flush=True)
                print("=" * 65 + "\n", flush=True)

        except Exception as e:
            print(f"\n❌ Mesh Runtime Execution Error Block: {e}\n", flush=True)


In [ ]:
import random

thread_random = str(random.randint(100000, 999999999))

run_chat(
    Deva_Super_Agent_mesh,
    thread_id=thread_random
)

In [ ]:
# Example prior run transcript removed for a clean production notebook.
# Run the cell above (run_chat) to start a fresh interactive session.
